# HazardFuse: pixel–patch fusion network for leached-metal hazard mapping (Glen Lyn)

This notebook trains **HazardFuse**, a supervised network that fuses a **center-pixel band-attention branch** with a **patch spatial-embedding branch** through a learned gate, to map higher and lower hazard index (HI) classes from multisource satellite features. It compares HazardFuse with ten benchmarks: Random Forest, XGBoost, Patch CNN, U-Net, U-Net++, Attention U-Net, ResU-Net, SE-ResU-Net, DeepLabv3+ and SegFormer.

**Evaluation design**
- **Split.** Buffered spatial block split (32-pixel blocks; 70/15/15% training/validation/test; 8-pixel buffer). Test blocks are never used for fitting or model selection.
- **Common protocol.** Every model uses the same input bands, standardisation (training-pixel statistics), class weights, early stopping on validation loss and threshold (0.5).
- **Seed repeats.** Every model is refitted with each seed in `SEEDS`; tables give the mean ± SD over seeds. Maps show the first seed (42).
- **Uncertainty.** A paired block bootstrap over the test blocks gives confidence intervals for the differences from HazardFuse.
- **Ablations.** HazardFuse components (architecture ablation) and input feature groups (feature-group ablation), both over seeds.
- **Export and transfer.** Every fitted model is exported with a manifest and can be applied to a new site.

**Sections:** 1 Configuration · 2 Setup · 3 Data · 4 Spatial split · 5 Map and figure utilities · 6 Models · 7 Training with seed repeats · 8 Model comparison · 9 Band attention · 10 Ablations · 11 Export and transfer

Outputs (figures as PDF, SVG and 300-dpi PNG; tables as CSV; GeoTIFFs; models) are written to `OUTPUT_DIR`. Set `SMOKE_TEST = True` to check the whole notebook in a few minutes before the full run.

> Repository note: this is the notebook of the published run (all 11 models, ablations, transfer); outputs are cleared. The HazardFuse model alone can be reproduced from the command line with `scripts/train.py` (see the README).

## 1. Configuration
Edit the paths and settings here only.
- `BAND_RENAME` is applied to the band names of every raster when it is read (here `CSTV_NDVI_Diff` → `CSTV_NDVI`), so rasters with old or new names both work, including at a transfer site.
- `WRITE_RENAMED_RASTERS = True` also writes `<name>_renamed.tif` with the new names stored in the file (band descriptions only; pixel values unchanged).
- `DROP_BANDS` removes QA60 (cloud flag) from the input of every model; all other bands, CSTV included, are used.

In [ ]:
import sys
from pathlib import Path

SITE = "Glen Lyn"
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR   = Path("/content/drive/MyDrive/CCR_Yearly_Feature_Stacks")             # EDIT
else:
    DATA_DIR = Path("../data/raw")                                       # EDIT (repository: data/raw)
OUTPUT_DIR = DATA_DIR / "HazardFuse_glen_lyn"
FEATURE_STEM, LABEL_STEM = "glen_lyn_Features", "glen_lyn_Label"

# ---- reproducibility and decision rule
BASE_SEED = 42
SEEDS = (42, 43, 44)                      # first seed = main run; all models are refitted with every seed
THRESHOLD = 0.5
USE_CLASS_WEIGHTS = True

# ---- buffered spatial block split
BLOCK_SIZE, SPLIT_FRAC, BUFFER_PX, N_SPLIT_TRIALS = 32, (0.70, 0.15, 0.15), 8, 500

# ---- band names: rules applied to every raster on reading (regex -> replacement), e.g. CSTV_NDVI_Diff -> CSTV_NDVI
BAND_RENAME = {r"^(CSTV_.+?)_Diff$": r"\1"}
WRITE_RENAMED_RASTERS = True            # True: also store the new names in a copy of the feature raster (section 3)

# ---- input bands: removed for EVERY model, so all models see the same stack
# DROP_BANDS = ["QA60"]                     # cloud-mask flag (not a surface property)
CSTV_BANDS = ["CSTV_NDVI_Diff", "CSTV_MTCI_Diff", "CSTV_SAVI_Diff", "CSTV_CIRE_Diff", "CSTV_NDRE1_Diff",
              "CSTV_NDRE2_Diff", "CSTV_LAI_Diff", "CSTV_LCC_Diff", "CSTV_CCC_Diff"]
DROP_BANDS = ["QA60"] #+ CSTV_BANDS        # cloud-mask flag; CSTV removed after the feature-group ablation
# ---- proposed model
PROPOSED = "HazardFuse"
HAZARDFUSE = dict(dropout=0.30, band_attention=True, center_branch=True, spatial_branch=True, fusion_gate=True)

# ---- training
PATIENCE = 20                             # early stopping on validation loss
PATCH, PATCH_EPOCHS, PATCH_BATCH, PATCH_LR = 21, 100, 256, 1e-3          # HazardFuse and Patch CNN (center pixel)
SEG_PATCH, SEG_STRIDE, MIN_LABEL_FRAC = 32, 8, 0.25                       # segmentation networks
SEG_EPOCHS, SEG_BATCH, SEG_LR, SEG_FILTERS = 100, 16, 1e-3, (32, 64, 128)
SEG_TTA = True                            # 8 dihedral transforms averaged at inference, for every segmentation network
RF_TREES = 100
XGB_PARAMS = dict(n_estimators=300, max_depth=8, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                  tree_method="hist", eval_metric="logloss", n_jobs=-1)

# ---- experiments and outputs
RUN_SEED_REPEATS, RUN_ARCH_ABLATION, RUN_FEATURE_ABLATION = True, True, True
ABLATION_SEEDS = SEEDS
N_BOOT = 2000                             # block-bootstrap resamples of the test blocks
SAVE_GEOTIFFS, EXPORT_MODELS = True, True

# ---- transfer to a new site (section 11): same band names as this site; labels are optional
TRANSFER_MODELS = (PROPOSED,)             # any keys of MODELS; every one is exported
NEW_SITE_NAME = "Bremo"
NEW_SITE_FEATURES = DATA_DIR / "bremo_Features.tif"               # EDIT, e.g. DATA_DIR / "bremo_Features.tif"
NEW_SITE_LABELS = DATA_DIR / "bremo_Label.tif"               # EDIT (optional), e.g. DATA_DIR / "bremo_Label.tif"
TRANSFER_STANDARDISE = "target"           # "source": Glen Lyn mean/SD; "target": statistics of the new site
TRANSFER_ADABN = True                     # re-estimate BatchNorm statistics on the new site (no labels used)
TRANSFER_THRESHOLD = "source"             # "source" (0.5), "prevalence" (match the training class-1 share) or a number

SMOKE_TEST = False                        # True: 2 epochs, one seed - checks the whole notebook in minutes
if SMOKE_TEST:
    PATCH_EPOCHS = SEG_EPOCHS = 2
    SEEDS = ABLATION_SEEDS = (BASE_SEED,)
    N_BOOT = 200

FIG_DIR, TAB_DIR, TIF_DIR, EXPORT_DIR, CKPT_DIR = (OUTPUT_DIR / d for d in
                                                   ("figures", "tables", "geotiffs", "exported_models", "checkpoints"))
for d in (FIG_DIR, TAB_DIR, TIF_DIR, EXPORT_DIR, CKPT_DIR):
    d.mkdir(parents=True, exist_ok=True)

## 2. Setup: libraries, figure style, raster input and metrics
Figures use Times New Roman (installed on Colab if missing), black text and a black border on every panel.

In [ ]:
import os, re, sys, gc, json, time, random, warnings, subprocess
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib.colors import ListedColormap, LinearSegmentedColormap, Normalize
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from IPython.display import display, Image

import joblib
import rasterio
from scipy.ndimage import distance_transform_edt
from sklearn.ensemble import RandomForestClassifier
try:
    from xgboost import XGBClassifier
except ImportError:
    XGBClassifier = None
    print("xgboost is not installed (pip install xgboost); XGBoost will be skipped.")
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_recall_fscore_support, roc_auc_score,
                             roc_curve, precision_recall_curve, confusion_matrix, matthews_corrcoef,
                             average_precision_score, brier_score_loss, log_loss, jaccard_score)

import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
AUTOTUNE = tf.data.AUTOTUNE
print("TensorFlow", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))


def set_seed(seed):
    """Seed Python, NumPy and TensorFlow (tf.keras.utils.set_random_seed covers all three)."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed); np.random.seed(seed)
    tf.keras.utils.set_random_seed(int(seed))


set_seed(BASE_SEED)

# ---- figure style: Times New Roman, black text, black border on every axes
if sys.platform.startswith("linux") and not any(f.name == "Times New Roman" for f in fm.fontManager.ttflist):
    for cmd in ("echo ttf-mscorefonts-installer msttcorefonts/accepted-mscorefonts-eula select true | debconf-set-selections",
                "DEBIAN_FRONTEND=noninteractive apt-get -qq install -y ttf-mscorefonts-installer fonts-liberation"):
        try:
            subprocess.run(cmd, shell=True, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, timeout=300)
        except Exception:
            pass
    for f in fm.findSystemFonts():
        try:
            fm.fontManager.addfont(f)
        except Exception:
            pass
plt.rcParams.update({"font.family": "serif",
                     "font.serif": ["Times New Roman", "Times", "Liberation Serif", "Nimbus Roman", "DejaVu Serif"],
                     "mathtext.fontset": "stix", "font.size": 9, "pdf.fonttype": 42, "svg.fonttype": "none",
                     "text.color": "black", "axes.labelcolor": "black", "axes.titlecolor": "black",
                     "axes.edgecolor": "black", "axes.linewidth": 0.8, "axes.spines.top": True, "axes.spines.right": True,
                     "xtick.color": "black", "ytick.color": "black", "axes.unicode_minus": False})
print("Figure font:", fm.FontProperties(family="serif").get_name())
GRID, C_PROP = "#e6e5e0", "#2a78d6"
CLS_COLORS = ["#008000", "#d73027"]                                   # lower HI (0), higher HI (1)
CLASS_LABELS = ("Lower HI (0)", "Higher HI (1)")
ERR_COLORS = ["#4575b4", "#fdae61", "#d73027", "#1a9641", "#e0e0e0"]   # TN, FP, FN, TP, not evaluated
ERR_LABELS = ["TN", "FP", "FN", "TP", "Not evaluated"]
SPLIT_COLORS = ["#a6d96a", "#d9d9d9", "#fdae61", "#7b3294"]
SPLIT_LABELS = ["Training", "Buffer", "Validation", "Test"]


def _blacken(fig):
    """Black text and a black border (all four spines) on every axes that is switched on."""
    for t in fig.findobj(mpl.text.Text):
        t.set_color("black")
    for a in fig.axes:
        if not a.axison:
            continue
        a.tick_params(colors="black", labelcolor="black")
        for s in a.spines.values():
            s.set_visible(True); s.set_edgecolor("black")
            if s.get_linewidth() < 0.6:
                s.set_linewidth(0.8)


def _save(fig, stem, out=None):
    out = FIG_DIR if out is None else out
    _blacken(fig)
    for ext in ("pdf", "svg"):
        fig.savefig(out / f"{stem}.{ext}", bbox_inches="tight", pad_inches=0.03)
    fig.savefig(out / f"{stem}.png", dpi=300, bbox_inches="tight", pad_inches=0.03)
    plt.close(fig)
    display(Image(filename=str(out / f"{stem}.png"), width=900))


# ---- raster input
def find_tif(folder, stem):
    for ext in (".tif", ".tiff", ".TIF", ".TIFF"):
        if (p := Path(folder) / f"{stem}{ext}").exists():
            return p
    raise FileNotFoundError(f"No {stem}.tif/.tiff in {folder}")


def canonical_name(name, rules=None):
    """Apply the BAND_RENAME rules to one band name (case-insensitive)."""
    for pattern, repl in (BAND_RENAME if rules is None else rules).items():
        name = re.sub(pattern, repl, name, flags=re.IGNORECASE)
    return name


def read_raster(path, rename=True):
    """All bands as float32 (bands, rows, cols) with nodata as NaN, the profile and the (renamed) band names."""
    with rasterio.open(path) as src:
        arr = src.read(masked=True).astype("float32").filled(np.nan)
        names = [d if d else f"band_{i + 1}" for i, d in enumerate(src.descriptions)]
        return arr, src.profile.copy(), [canonical_name(n) for n in names] if rename else names


def rename_raster_bands(src_path, dst_path=None, rules=None):
    """
    Store renamed band descriptions in a GeoTIFF. Only the band descriptions change; pixel values, nodata,
    CRS and transform are untouched. dst_path=None edits the file in place; otherwise a copy is renamed.
    Returns {old name: new name} for the bands that changed.
    """
    import shutil
    target = Path(src_path) if dst_path is None else Path(dst_path)
    if dst_path is not None:
        shutil.copy2(src_path, target)
    changed = {}
    with rasterio.open(target, "r+") as dst:
        for i, old in enumerate(dst.descriptions, start=1):
            new = canonical_name(old, rules) if old else old
            if new != old:
                dst.set_band_description(i, new); changed[old] = new
    return changed


# ---- metrics
def kappa_se(cm):
    """Large-sample standard error of Cohen's kappa (Fleiss, Cohen and Everitt, 1969)."""
    p = np.asarray(cm, dtype=float)
    n = p.sum()
    if n == 0:
        return np.nan
    p /= n
    row, col = p.sum(1), p.sum(0)
    po, pe = np.trace(p), float(row @ col)
    if np.isclose(pe, 1.0):
        return np.nan
    k = (po - pe) / (1 - pe)
    off = p.copy(); np.fill_diagonal(off, 0.0)
    var = (np.sum(np.diag(p) * (1 - (row + col) * (1 - k)) ** 2) + (1 - k) ** 2 * np.sum(off * (col[None] + row[:, None]) ** 2)
           - (k - pe * (1 - k)) ** 2) / (n * (1 - pe) ** 2)
    return float(np.sqrt(var)) if var > 0 else 0.0


def compute_metrics(y_true, prob, threshold=THRESHOLD):
    y, p = np.asarray(y_true).astype(int), np.asarray(prob, dtype=float)
    pred = (p >= threshold).astype(int)
    prec, rec, f1, sup = precision_recall_fscore_support(y, pred, labels=[0, 1], zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    two = len(np.unique(y)) == 2
    n = tn + fp + fn + tp
    pe = ((tp + fp) * (tp + fn) + (fn + tn) * (fp + tn)) / (n * n)
    return {"Accuracy": accuracy_score(y, pred), "Balanced_Acc": balanced_accuracy_score(y, pred),
            "Precision_0": prec[0], "Recall_0": rec[0], "F1_0": f1[0],
            "Precision_1": prec[1], "Recall_1": rec[1], "F1_1": f1[1], "Macro_F1": f1.mean(),
            "Kappa": ((tp + tn) / n - pe) / (1 - pe) if not np.isclose(pe, 1) else np.nan,
            "Kappa_SE": kappa_se([[tn, fp], [fn, tp]]), "MCC": matthews_corrcoef(y, pred),
            "IoU_0": jaccard_score(y, pred, pos_label=0, zero_division=0), "IoU_1": tp / (tp + fp + fn) if tp + fp + fn else np.nan,
            "ROC_AUC_1": roc_auc_score(y, p) if two else np.nan, "PR_AUC_1": average_precision_score(y, p) if two else np.nan,
            "PR_AUC_0": average_precision_score(1 - y, 1 - p) if two else np.nan,
            "Brier": brier_score_loss(y, np.clip(p, 0, 1)), "Log_Loss": log_loss(y, np.clip(p, 1e-7, 1 - 1e-7), labels=[0, 1]),
            "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp), "N": int(n)}


def metrics_from_counts(tn, fp, fn, tp):
    """Vectorised threshold metrics from confusion counts (used by the block bootstrap)."""
    tn, fp, fn, tp = (np.asarray(v, dtype=float) for v in (tn, fp, fn, tp))
    n = tn + fp + fn + tp
    with np.errstate(divide="ignore", invalid="ignore"):
        rec1, rec0 = tp / (tp + fn), tn / (tn + fp)
        f1_1, f1_0 = 2 * tp / (2 * tp + fp + fn), 2 * tn / (2 * tn + fn + fp)
        po, pe = (tp + tn) / n, ((tp + fp) * (tp + fn) + (tn + fn) * (tn + fp)) / n ** 2
        mcc = (tp * tn - fp * fn) / np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    return {"Balanced_Acc": (rec0 + rec1) / 2, "Macro_F1": (f1_0 + f1_1) / 2, "Kappa": (po - pe) / (1 - pe),
            "MCC": np.nan_to_num(mcc), "IoU_1": tp / (tp + fp + fn)}

## 3. Data
Bands are matched by name (after `BAND_RENAME`), so the band order of the raster does not matter. Bands that belong to no feature group are reported. `rename_raster_bands(path)` (section 2) renames the bands of any GeoTIFF in place, or of a copy if a second path is given.

In [ ]:
FEATURE_FILE, LABEL_FILE = find_tif(DATA_DIR, FEATURE_STEM), find_tif(DATA_DIR, LABEL_STEM)
if WRITE_RENAMED_RASTERS:                 # writes <name>_renamed.tif next to the original; the original is not modified
    _new = FEATURE_FILE.with_name(f"{FEATURE_FILE.stem}_renamed.tif")
    _chg = rename_raster_bands(FEATURE_FILE, _new)
    print(f"Wrote {_new.name}: {len(_chg)} bands renamed", _chg)
_arr, profile, RAW_NAMES = read_raster(FEATURE_FILE)
X_RAW = np.transpose(_arr, (1, 2, 0))
Y_raw = read_raster(LABEL_FILE)[0][0]
assert X_RAW.shape[:2] == Y_raw.shape, "feature and label grids differ"
ROWS, COLS, N_RAW = X_RAW.shape
valid = np.isin(Y_raw, [0, 1]) & np.isfinite(X_RAW).any(axis=-1)
Y = np.where(valid, Y_raw, 0).astype(np.uint8)

# ---- feature families (bands absent from the raster are skipped)
FEATURE_GROUPS = {
    "VI": ["NDVI", "MTCI", "SAVI", "CIRE", "NDRE1", "NDRE2", "LAI", "LCC", "CCC"],
    "S2": ["B1", "B2", "B3", "B4", "B5", "B6", "B7", "B8", "B8A", "B9", "B11", "B12"],
    "Land Cover": ["DW_Vegetation", "DW_LandCover", "DW_Snow", "DW_Built", "DW_Shrubs", "DW_Bare", "DW_Crops",
                   "DW_FloodedVeg", "DW_Grass", "DW_Trees", "DW_Water"],
#     "CSTV": CSTV_BANDS,
    "CSTV": ["CSTV_NDVI", "CSTV_MTCI", "CSTV_SAVI", "CSTV_CIRE", "CSTV_NDRE1", "CSTV_NDRE2", "CSTV_LAI", "CSTV_LCC", "CSTV_CCC"],
    "Topographic": ["DEM", "Slope_deg", "Aspect_deg", "Curvature"],
    "Soil": ["Moisture", "pH", "Sand_pct_0_5cm", "Silt_pct_0_5cm", "Clay_pct_0_5cm"],
    "Climate": ["Temperature", "Humidity"],
    "Hydrologic": ["PrecipJJA_mm_mean", "UpstreamArea_km2", "HAND_m", "TWI"],
    "RoC": ["NDVI_Rate", "MTCI_Rate", "SAVI_Rate", "CIRE_Rate", "NDRE1_Rate", "NDRE2_Rate", "LAI_Rate", "LCC_Rate", "CCC_Rate"],
    "Quality": ["QA60"],
}
assert len(set(n.lower() for n in RAW_NAMES)) == len(RAW_NAMES), "duplicate band names after renaming"
_pos = {n.lower(): i for i, n in enumerate(RAW_NAMES)}
GROUP_IDX = {g: [_pos[b.lower()] for b in dict.fromkeys(bands) if b.lower() in _pos] for g, bands in FEATURE_GROUPS.items()}
_drop = {b.lower() for b in DROP_BANDS}
IN_IDX = [i for i, n in enumerate(RAW_NAMES) if n.lower() not in _drop]                 # model input
IN_NAMES = [RAW_NAMES[i] for i in IN_IDX]
_grouped = {i for idx in GROUP_IDX.values() for i in idx}

print(f"Features {X_RAW.shape} | valid pixels {valid.sum():,} ({100 * valid.mean():.1f}%) | class-1 fraction {Y[valid].mean():.3f}")
print(f"Model input: {len(IN_IDX)} of {N_RAW} bands (removed: {[RAW_NAMES[i] for i in range(N_RAW) if i not in IN_IDX]})")
display(pd.DataFrame({"Bands in raster": {g: len(i) for g, i in GROUP_IDX.items()},
                      "In model input": {g: sum(k in IN_IDX for k in i) for g, i in GROUP_IDX.items()}}))
if len(_grouped) < N_RAW:
    print("Bands in no family:", [RAW_NAMES[i] for i in range(N_RAW) if i not in _grouped])

## 4. Spatial split and standardisation
Blocks are assigned to training, validation and test by a random search (500 trials) that matches the target fractions and the site class balance. Training and validation pixels within 8 pixels of a test block, and training pixels within 8 pixels of a validation block, form the buffer and are not used.

In [ ]:
n_block_rows, n_block_cols = int(np.ceil(ROWS / BLOCK_SIZE)), int(np.ceil(COLS / BLOCK_SIZE))
rr, cc = np.indices((ROWS, COLS))
block_id = (rr // BLOCK_SIZE) * n_block_cols + (cc // BLOCK_SIZE)
n_blocks = n_block_rows * n_block_cols
blk_n = np.bincount(block_id[valid], minlength=n_blocks).astype(float)
blk_pos = np.bincount(block_id[valid], weights=Y[valid], minlength=n_blocks)
candidates = np.flatnonzero(blk_n > 0)
n_valid, p_all = valid.sum(), Y[valid].mean()


def propose_split(rng):
    order = rng.permutation(candidates)
    frac = blk_n[order] / n_valid
    mid = np.cumsum(frac) - frac / 2
    assign = np.full(n_blocks, -1, dtype=int)
    assign[order] = np.where(mid < SPLIT_FRAC[2], 2, np.where(mid < SPLIT_FRAC[2] + SPLIT_FRAC[1], 1, 0))
    return assign


def split_score(assign):
    s = 0.0
    for code, target in ((2, SPLIT_FRAC[2]), (1, SPLIT_FRAC[1]), (0, SPLIT_FRAC[0])):
        sel = assign == code
        n, pos = blk_n[sel].sum(), blk_pos[sel].sum()
        if n == 0 or pos == 0 or pos == n:                    # every subset needs both classes
            return np.inf
        s += abs(n / n_valid - target) + abs(pos / n - p_all)
    return s


_rng = np.random.default_rng(BASE_SEED)
best, best_s = None, np.inf
for _ in range(N_SPLIT_TRIALS):
    cand = propose_split(_rng)
    if (s_ := split_score(cand)) < best_s:
        best, best_s = cand, s_
if best is None:
    raise RuntimeError("No split contains both classes in every subset; change BLOCK_SIZE or N_SPLIT_TRIALS.")
block_split = best[block_id]                                  # -1 empty, 0 train, 1 validation, 2 test
train_mask = valid & (block_split == 0) & (distance_transform_edt(~np.isin(block_split, [1, 2])) > BUFFER_PX)
val_mask = valid & (block_split == 1) & (distance_transform_edt(block_split != 2) > BUFFER_PX)
test_mask = valid & (block_split == 2)
buffer_mask = valid & ~(train_mask | val_mask | test_mask)
MASKS = {"train": train_mask, "val": val_mask, "test": test_mask}

T0 = pd.DataFrame({k: {"Pixels": int(m.sum()), "Share (%)": 100 * m.sum() / n_valid, "Class-1 fraction": Y[m].mean()}
                   for k, m in (("Training", train_mask), ("Buffer", buffer_mask), ("Validation", val_mask), ("Test", test_mask))}).T
T0.to_csv(TAB_DIR / "T0_split.csv")
display(T0.round(3))
print("Test blocks:", len(np.unique(block_id[test_mask])))

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=RuntimeWarning)              # all-NaN bands
    BAND_MEAN, BAND_STD = np.nanmean(X_RAW[train_mask], axis=0), np.nanstd(X_RAW[train_mask], axis=0)
BAND_MEAN = np.where(np.isfinite(BAND_MEAN), BAND_MEAN, 0.0)
BAND_STD = np.where(np.isfinite(BAND_STD) & (BAND_STD > 1e-12), BAND_STD, 1.0)
X_ALL = np.nan_to_num((X_RAW - BAND_MEAN) / BAND_STD, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)   # NaN -> training mean
X_ALL[~valid] = 0.0
X_IN = X_ALL[..., IN_IDX]                                                  # common input of every model

n_tr, n_pos = int(train_mask.sum()), int(Y[train_mask].sum())
W_NEG, W_POS = (n_tr / (2 * (n_tr - n_pos)), n_tr / (2 * n_pos)) if USE_CLASS_WEIGHTS else (1.0, 1.0)
print(f"Training pixels {n_tr:,} (class 0 {n_tr - n_pos:,}, class 1 {n_pos:,}) | class weights {W_NEG:.3f}, {W_POS:.3f}")

## 5. Map and figure utilities

In [ ]:
def slug(name):
    return re.sub(r"[^A-Za-z0-9]+", "_", name.replace("+", "plus")).strip("_")


def class_and_error_maps(prob, labels=None, valid_mask=None, scope=None, threshold=THRESHOLD):
    """Class map (0, 1, 255 = nodata) and error map (0 TN, 1 FP, 2 FN, 3 TP, 4 valid but not scored, 255 nodata)."""
    labels = Y if labels is None else labels
    valid_mask = valid if valid_mask is None else valid_mask
    scope = valid_mask if scope is None else scope
    pred, truth = np.nan_to_num(prob) >= threshold, labels.astype(bool)
    cls = np.full(prob.shape, 255, np.uint8); cls[valid_mask] = pred[valid_mask]
    err = np.full(prob.shape, 255, np.uint8); err[valid_mask] = 4
    for code, (t, p) in enumerate(((0, 0), (0, 1), (1, 0), (1, 1))):
        err[scope & (truth == t) & (pred == p)] = code
    return cls, err


def outline(ax, region=None, lw=0.8):
    region = (block_split == 2) if region is None else region
    ax.contour(region.astype(float), levels=[0.5], colors="k", linewidths=lw)


def show_prob(ax, prob, valid_mask=None):
    valid_mask = valid if valid_mask is None else valid_mask
    return ax.imshow(np.ma.masked_where(~valid_mask, prob), cmap="RdYlGn_r", vmin=0, vmax=1, interpolation="nearest")


def show_cls(ax, cls):
    ax.imshow(np.ma.masked_equal(cls, 255), cmap=ListedColormap(CLS_COLORS), vmin=0, vmax=1, interpolation="nearest")


def show_err(ax, err):
    ax.imshow(np.ma.masked_equal(err, 255), cmap=ListedColormap(ERR_COLORS), vmin=0, vmax=4, interpolation="nearest")


def err_handles(err=None):
    lab = [f"{l} ({int((err == i).sum()):,})" if err is not None and i < 4 else l for i, l in enumerate(ERR_LABELS)]
    return [Patch(facecolor=c, edgecolor="black", linewidth=0.4, label=l) for c, l in zip(ERR_COLORS, lab)]


def cls_handles():
    return [Patch(facecolor=c, edgecolor="black", linewidth=0.4, label=l) for c, l in zip(CLS_COLORS, CLASS_LABELS)]


def _map_axes(ax, title):
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(title, fontsize=10, fontweight="bold")


def write_geotiff(path, array, dtype, nodata, ref_profile=None):
    ref = profile if ref_profile is None else ref_profile
    meta = {"driver": "GTiff", "height": array.shape[0], "width": array.shape[1], "count": 1, "dtype": dtype,
            "crs": ref.get("crs"), "transform": ref.get("transform"), "nodata": nodata, "compress": "lzw"}
    with rasterio.open(path, "w", **meta) as dst:
        dst.write(array.astype(dtype), 1)


def save_map_geotiffs(name, prob, out=None, labels=None, valid_mask=None, ref_profile=None, threshold=THRESHOLD):
    """Probability, class and whole-area error map (when labels are known) as GeoTIFFs."""
    out = TIF_DIR if out is None else out
    valid_mask = valid if valid_mask is None else valid_mask
    known = labels is not None
    cls, err = class_and_error_maps(prob, labels if known else np.zeros(prob.shape, np.uint8), valid_mask,
                                    threshold=threshold)
    write_geotiff(out / f"{slug(name)}_probability.tif", np.where(valid_mask, prob, np.nan).astype(np.float32),
                  "float32", np.nan, ref_profile)
    write_geotiff(out / f"{slug(name)}_class.tif", cls, "uint8", 255, ref_profile)
    if known:
        write_geotiff(out / f"{slug(name)}_error.tif", err, "uint8", 255, ref_profile)

In [ ]:
plt.rcParams["axes.unicode_minus"] = True
MK = {"Kappa": ("Cohen's κ", "D", "#1a9e77", 34), "Macro_F1": ("Macro-F1", "o", "#2a78d6", 34),
      "Balanced_Acc": ("Balanced accuracy", "s", "#e8612c", 30), "ROC_AUC_1": ("ROC-AUC", "^", "#e6a100", 26),
      "Accuracy": ("Accuracy", "v", "#5b3fa8", 26), "IoU_1": ("IoU (class 1)", "D", "#1a9e77", 30),
      "PR_AUC_1": ("PR-AUC", "s", "#e8612c", 28), "MCC": ("MCC", "v", "#5b3fa8", 28)}
C0, C1, C_STEM, C_DL, C_ML = "#e8612c", "#2a78d6", "#b5b5ad", "#6b6b63", "#e8612c"
SHADE = "#e8eef8"


def _minus(v, fmt="+.3f"):
    return format(v, fmt).replace("-", "−")


def _rowgrid(ax, n):
    for i in range(n):
        ax.axhline(i, color=GRID, lw=0.6, zorder=0)
    ax.grid(axis="x", color=GRID, lw=0.6, zorder=0)
    ax.set_axisbelow(True)


def _lims(values, step=0.05, pad=0.005):
    v = np.asarray(values, float); v = v[np.isfinite(v)]
    return np.floor((v.min() - pad) / step) * step, min(1.0, np.ceil((v.max() + pad) / step) * step) + pad


# ---------------------------------------------------------------- model comparison (dot plot + dumbbells)
def fig_model_comparison(df, proposed=PROPOSED, stem="Fig_model_comparison"):
    """df: one row per model in display order (best first); columns Kappa, Macro_F1, Balanced_Acc, ROC_AUC_1,
    Recall_0, Recall_1, Precision_0, Precision_1 (means over seeds)."""
    n, y = len(df), np.arange(len(df))
    lo, hi = _lims(df[["Kappa", "Macro_F1", "Balanced_Acc", "ROC_AUC_1", "Recall_0", "Recall_1",
                       "Precision_0", "Precision_1"]].values.ravel())
    fig, axs = plt.subplots(1, 3, figsize=(10.2, 0.42 * n + 1.9), sharey=True, gridspec_kw={"width_ratios": [1.25, 1, 1]})
    for ax in axs:
        _rowgrid(ax, n)
        if proposed in df.index:
            k = list(df.index).index(proposed)
            ax.axhspan(k - 0.45, k + 0.45, color=SHADE, zorder=0)
        ax.set_xlim(lo, hi); ax.set_xlabel("Score")
    for key in ("Kappa", "Balanced_Acc", "Macro_F1", "ROC_AUC_1"):
        lab, m, c, s = MK[key]
        axs[0].scatter(df[key], y, marker=m, s=s, color=c, edgecolor="white", linewidth=0.5, zorder=3, label=lab)
    for ax, (a, b) in zip(axs[1:], (("Recall_0", "Recall_1"), ("Precision_0", "Precision_1"))):
        ax.hlines(y, df[a], df[b], color=C_STEM, lw=2.4, zorder=2)
        ax.scatter(df[a], y, s=36, color=C0, edgecolor="white", linewidth=0.5, zorder=3, label="Class 0 (lower HI)")
        ax.scatter(df[b], y, s=36, color=C1, edgecolor="white", linewidth=0.5, zorder=3, label="Class 1 (higher HI)")
    axs[0].set_yticks(y); axs[0].set_yticklabels(df.index, fontsize=10); axs[0].set_ylim(n - 0.5, -0.5)
    for t in axs[0].get_yticklabels():
        t.set_fontweight("bold" if t.get_text() == proposed else "normal")
    for ax, t in zip(axs, ("(a) Overall performance", "(b) Class-wise recall", "(c) Class-wise precision")):
        ax.set_title(t, loc="left", fontweight="bold", fontsize=12)
    fig.tight_layout(w_pad=0.6)
    p0, p1, p2 = (a.get_position() for a in axs)
    ybot = p0.y0 - 0.55 / fig.get_figheight()
    fig.legend(*axs[0].get_legend_handles_labels(), loc="upper center", bbox_to_anchor=((p0.x0 + p0.x1) / 2, ybot),
               ncol=2, frameon=False, handletextpad=0.3)
    fig.legend(*axs[1].get_legend_handles_labels(), loc="upper center", bbox_to_anchor=((p1.x0 + p2.x1) / 2, ybot),
               ncol=2, frameon=False, title="Panels (b) and (c)", title_fontsize=12)
    _save(fig, stem)


# ---------------------------------------------------------------- ROC and PR space
def _iso_levels(vals, step=0.05, k=4):
    v = np.asarray(vals, float)
    top = np.ceil(np.nanmax(v) / step) * step
    lv = np.round(top - step * np.arange(k + 2), 3)
    return [x for x in lv if 0 < x < 1][:k]


def _place_labels(ax, xs, ys, texts, colors=None, weights=None, gap=5):
    """Put each label at the free side of its point (largest clearance from points and earlier labels)."""
    fig = ax.figure; fig.canvas.draw()
    P = ax.transData.transform(np.c_[xs, ys]); dpi = fig.dpi / 72
    cands = [(gap, 0), (-gap, 0), (0, gap), (0, -gap), (gap, gap), (-gap, gap), (gap, -gap), (-gap, -gap)]
    taken = list(P)
    for i, t in enumerate(texts):
        best, bd = cands[0], -1
        for dx, dy in cands:
            q = P[i] + np.array([dx, dy]) * dpi * 1.4
            d = min(np.hypot(*(q - o)) for j, o in enumerate(taken) if j != i)
            if d > bd + 1e-6:
                best, bd = (dx, dy), d
        taken.append(P[i] + np.array(best) * dpi * 1.4)
        ha = "left" if best[0] > 0 else "right" if best[0] < 0 else "center"
        va = "bottom" if best[1] > 0 else "top" if best[1] < 0 else "center"
        ax.annotate(t, (xs[i], ys[i]), xytext=best, textcoords="offset points", ha=ha, va=va, fontsize=12,
                    color=(colors or {}).get(i, "black"), fontweight=(weights or {}).get(i, "normal"))


def fig_roc_pr_space(df, family, proposed=PROPOSED, stem="Fig_ROC_PR_space"):
    """df in rank order (by test Macro-F1); columns Recall_0, Recall_1, Precision_0, Precision_1.
    family: model -> 'Proposed' | 'ML' | 'DL'."""
    names = list(df.index); rank = {m: i + 1 for i, m in enumerate(names)}
    panels = [("(a) ROC space", 1 - df["Recall_0"].values, df["Recall_1"].values,
               "False-positive rate (1 − specificity)", "True-positive rate (sensitivity)", "J"),
              ("(b) PR space, class 1 (higher HI)", df["Recall_1"].values, df["Precision_1"].values, "Recall", "Precision", "F1"),
              ("(c) PR space, class 0 (lower HI)", df["Recall_0"].values, df["Precision_0"].values, "Recall", "Precision", "F1")]
    fig, axs = plt.subplots(1, 3, figsize=(10.2, 4.2))
    for ax, (title, x, y, xl, yl, iso) in zip(axs, panels):
        dx, dy = np.ptp(x) or 0.02, np.ptp(y) or 0.02
        x0, x1 = x.min() - 0.12 * dx, x.max() + 0.12 * dx
        y0, y1 = y.min() - 0.12 * dy, y.max() + 0.12 * dy
        if iso == "F1":
            x1 = min(x1, 1.005); y1 = min(y1, 1.005)
        ax.set_xlim(x0, x1); ax.set_ylim(y0, y1)
        xx = np.linspace(x0, x1, 400)
        if iso == "J":
            for J in _iso_levels(y - x):
                ax.plot(xx, xx + J, ls="--", lw=0.7, color="#a8a8a0", zorder=1)
                xt = min(x1, y1 - J)                           # where the line leaves the panel
                if x0 + 0.15 * (x1 - x0) < xt <= x1:          # skip labels that would cross the left border
                    ax.annotate(f"J = {J:.2f}", (xt, xt + J), xytext=(-3, -2), textcoords="offset points",
                                ha="right", va="top", fontsize=7.5)
        else:
            f1 = 2 * x * y / (x + y)
            for F in _iso_levels(f1, step=0.02 if np.ptp(f1) < 0.08 else 0.05):
                r = xx[xx > F / 2 + 1e-6]; p = F * r / (2 * r - F)
                ok = (p >= y0) & (p <= y1)
                if ok.sum() > 1:
                    ax.plot(r[ok], p[ok], ls="--", lw=0.7, color="#a8a8a0", zorder=1)
                    ax.annotate(f"F1 = {F:.2f}", (r[ok][0], p[ok][0]), xytext=(3, -2), textcoords="offset points",
                                ha="left", va="top", fontsize=7.5)
        for i, m in enumerate(names):
            fam = family.get(m, "DL")
            if fam == "Proposed":
                ax.scatter(x[i], y[i], marker="*", s=170, color=C1, edgecolor="white", linewidth=0.4, zorder=4)
            else:
                ax.scatter(x[i], y[i], marker="s" if fam == "ML" else "o", s=38, color=C_ML if fam == "ML" else C_DL,
                           edgecolor="white", linewidth=0.4, zorder=3)
        k = names.index(proposed) if proposed in names else -1
        _place_labels(ax, x, y, [str(rank[m]) for m in names], colors={k: C1}, weights={k: "bold"})
        ax.set_title(title, fontweight="bold", fontsize=12); ax.set_xlabel(xl); ax.set_ylabel(yl)
        ax.grid(color=GRID, lw=0.6); ax.set_axisbelow(True)
    handles = [Line2D([], [], marker="*", ls="", ms=11, color=C1, label=f"{proposed} (proposed)"),
               Line2D([], [], marker="s", ls="", ms=6, color=C_ML, label="Machine-learning benchmark"),
               Line2D([], [], marker="o", ls="", ms=6, color=C_DL, label="Deep-learning benchmarks"),
               Line2D([], [], ls="--", lw=0.8, color="#a8a8a0", label="Iso-lines: Youden's J (a), F1 (b, c)")]
    fig.tight_layout(w_pad=1.0)
    fig.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, 0.02), ncol=4, frameon=False, columnspacing=1.2)
    key = [f"{rank[m]} {m}" for m in names]
    lines = ["   ".join(key[i:i + 6]) for i in range(0, len(key), 6)]
    fig.text(0.5, -0.06, "Model key (ranked by test Macro-F1)\n" + "\n".join(lines), ha="center", va="top", fontsize=11)
    _save(fig, stem)


# ---------------------------------------------------------------- lollipop ablations
def _lollipop(ax, delta, metrics, row_labels, band=None, style=None):
    n, k = len(delta), len(metrics)
    off = np.linspace(0.27, -0.27, k)                        # first metric drawn at the top of each row
    _rowgrid(ax, n)
    if band:
        ax.axvspan(-band, band, color="#e8e6df", zorder=0)
    for j, key in enumerate(metrics):
        lab, m, c, s = {**MK, **(style or {})}[key]
        yy = np.arange(n) - off[j]
        ax.hlines(yy, 0, delta[key], color=c, alpha=0.55, lw=1.6, zorder=2)
        ax.scatter(delta[key], yy, marker=m, s=s, color=c, edgecolor="white", linewidth=0.4, zorder=3, label=lab)
    ax.axvline(0, color="black", lw=1.0, zorder=2)
    ax.set_yticks(np.arange(n)); ax.set_yticklabels(row_labels, fontsize=12); ax.set_ylim(n - 0.5, -0.85)
    v = delta[metrics].values
    lo, hi = np.nanmin(np.r_[v.ravel(), 0]), np.nanmax(np.r_[v.ravel(), 0])
    span = max(hi - lo, 0.01)
    ax.set_xlim(lo - 0.06 * span, hi + 0.12 * span + (band or 0))


def fig_arch_ablation(delta, params, full_params, stem="Fig_architecture_ablation"):
    """delta: rows = variants, columns Accuracy, Macro_F1, IoU_1, ROC_AUC_1, PR_AUC_1 (variant − full, mean over seeds).
    params: trainable parameters per variant; full_params: parameters of the full model."""
    delta = delta.sort_values("Macro_F1", ascending=False)
    params = pd.Series(params).reindex(delta.index)
    labels = [f"{v}\nΔMacro-F1 = {_minus(d)}" for v, d in delta["Macro_F1"].items()]
    fig, (a, b) = plt.subplots(1, 2, figsize=(10.2, 0.8 * len(delta) + 1.5), sharey=True,
                               gridspec_kw={"width_ratios": [2.6, 1]})
    _lollipop(a, delta, ["PR_AUC_1", "ROC_AUC_1", "IoU_1", "Macro_F1", "Accuracy"], labels)
    a.text(0, -0.68, "← performance loss  ", ha="right", va="center", fontsize=11)
    a.set_xlabel("Change relative to full model (variant − full)")
    a.set_title("(a) Performance change after removing each component", fontweight="bold", fontsize=12, loc="left")
    k = params.values / 1e3
    _rowgrid(b, len(delta))
    b.barh(np.arange(len(delta)), k, height=0.5, color="#d9d8d2", zorder=2)
    for i, v in enumerate(k):
        b.text(0.02 * full_params / 1e3, i, f"{v:.1f} k ({100 * v * 1e3 / full_params:.0f}%)", va="center", fontsize=11)
    b.axvline(full_params / 1e3, color="black", ls="--", lw=1.0)
    b.text(full_params / 1e3, -0.68, f" full: {full_params / 1e3:.1f} k", ha="left", va="center", fontsize=11)
    b.set_xlim(0, full_params / 1e3 * 1.45); b.set_xlabel("Trainable parameters (thousands)")
    b.set_title("(b) Model size", fontweight="bold", fontsize=12, loc="left")
    fig.tight_layout(w_pad=0.8)
    h, l = a.get_legend_handles_labels()
    a.legend(h[::-1], l[::-1], loc="upper center", bbox_to_anchor=(0.7, -0.13), ncol=5, frameon=False, handletextpad=0.2)
    _save(fig, stem)


def fig_feature_ablation(delta, recall, full_recall, noise=None, stem="Fig_feature_ablation"):
    """delta: rows = configurations, columns Macro_F1, Balanced_Acc, Kappa, IoU_1 (variant − full).
    recall: Recall_0, Recall_1 per configuration; full_recall: (recall class 0, recall class 1) of the full set;
    noise: half-width of the shaded band (SD of the full-set Macro-F1 over seeds)."""
    delta = delta.sort_values("Macro_F1", ascending=False)
    recall = recall.reindex(delta.index)
    labels = [f"{v}\nΔMacro-F1 = {_minus(d)}" for v, d in delta["Macro_F1"].items()]
    fig, (a, b) = plt.subplots(1, 2, figsize=(10.2, 0.5 * len(delta) + 1.6), sharey=True,
                               gridspec_kw={"width_ratios": [2.1, 1]})
    _lollipop(a, delta, ["IoU_1", "Balanced_Acc", "Kappa", "Macro_F1"], labels, band=noise,
              style={"IoU_1": ("IoU (class 1)", "v", "#5b3fa8", 28)})
    a.text(0, -0.68, "← loss | gain →", ha="center", va="center", fontsize=11)
    a.set_xlabel("Change relative to full feature set (variant − full)")
    a.set_title("(a) Change after removing each feature group", fontweight="bold", fontsize=11, loc="left")
    y = np.arange(len(delta))
    _rowgrid(b, len(delta))
    b.hlines(y, recall["Recall_0"], recall["Recall_1"], color=C_STEM, lw=2.4, zorder=2)
    b.scatter(recall["Recall_0"], y, s=36, color=C0, edgecolor="white", linewidth=0.5, zorder=3)
    b.scatter(recall["Recall_1"], y, s=36, color=C1, edgecolor="white", linewidth=0.5, zorder=3)
    b.axvline(full_recall[0], color=C0, ls="--", lw=0.9); b.axvline(full_recall[1], color=C1, ls="--", lw=0.9)
    lo, hi = _lims(np.r_[recall.values.ravel(), full_recall], step=0.05)
    b.set_xlim(lo, min(hi, 1.005)); b.set_xlabel("Recall")
    b.set_title("(b) Class-wise recall", fontweight="bold", fontsize=10, loc="left")
    fig.tight_layout(w_pad=0.8)
    h, l = a.get_legend_handles_labels()
    h, l = h[::-1], l[::-1]
    if noise:
        h.append(Patch(color="#e8e6df")); l.append(f"±{noise:.3f} (SD of the full-set\nMacro-F1 over seeds)")
    a.legend(h, l, loc="upper center", bbox_to_anchor=(0.5, -0.11), ncol=3, frameon=False, handletextpad=0.3)
    b.legend(handles=[Line2D([], [], marker="o", ls="", color=c, ms=7, label=t)
                      for c, t in ((C0, "Class 0 (lower HI)"), (C1, "Class 1 (higher HI)"))],
             loc="upper center", bbox_to_anchor=(0.5, -0.11), ncol=2, frameon=False,
             title="Dashed lines: full feature set", title_fontsize=12)
    _save(fig, stem)


# ---------------------------------------------------------------- other summary figures
def fig_split_performance(runs, name=PROPOSED, stem=None):
    """Grouped bars: training, validation and test scores (mean ± SD over seeds) for one model."""
    keys = ["MCC", "Macro_F1", "Balanced_Acc", "ROC_AUC_1", "PR_AUC_1"]
    labs = ["MCC", "Macro-F1", "Balanced\naccuracy", "ROC-AUC", "PR-AUC\n(class 1)"]
    fig, ax = plt.subplots(figsize=(7.2, 3.4))
    w = 0.26
    for j, (split, lab, c) in enumerate((("train", "Training", SPLIT_COLORS[0]), ("val", "Validation", SPLIT_COLORS[2]),
                                         ("test", "Test", SPLIT_COLORS[3]))):
        mu, sd = summarise(runs, split, keys)
        x = np.arange(len(keys)) + (j - 1) * w
        ax.bar(x, mu, w, yerr=None if sd.isna().all() else sd, color=c, edgecolor="black", linewidth=0.5,
               capsize=2, error_kw={"lw": 0.8}, label=lab, zorder=2)
        for xi, v, e in zip(x, mu, sd.fillna(0)):
            ax.text(xi, v + e + 0.015, f"{v:.3f}", ha="center", va="bottom", fontsize=11, rotation=90)
    ax.set_xticks(np.arange(len(keys))); ax.set_xticklabels(labs)
    ax.set_ylim(0, 1.2); ax.set_ylabel("Score"); ax.grid(axis="y", color=GRID, lw=0.6); ax.set_axisbelow(True)
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=3, frameon=False)
    ax.set_title(f"{name}: training, validation and test performance (mean ± SD over seeds)", fontsize=12, fontweight="bold")
    fig.tight_layout()
    _save(fig, stem or f"Fig_{slug(name)}_split_performance")


def fig_bootstrap(boot, proposed=PROPOSED, stem="Fig_block_bootstrap"):
    """Forest plot of benchmark − proposed differences on the test blocks (median and 95% interval)."""
    boot = boot.sort_values("dMacro_F1")
    y = np.arange(len(boot))
    fig, axs = plt.subplots(1, 2, figsize=(8.6, 0.38 * len(boot) + 1.3), sharey=True)
    for ax, key, lab in zip(axs, ("dMacro_F1", "dMCC"), ("ΔMacro-F1", "ΔMCC")):
        _rowgrid(ax, len(boot))
        ax.hlines(y, boot[f"{key}_lo"], boot[f"{key}_hi"], color="black", lw=1.1, zorder=2)
        ax.scatter(boot[key], y, s=30, color=C1, edgecolor="black", linewidth=0.5, zorder=3)
        ax.axvline(0, color="black", ls="--", lw=0.9)
        ax.set_xlabel(f"{lab} (benchmark − {proposed})")
    axs[0].set_yticks(y); axs[0].set_yticklabels(boot.index); axs[0].set_ylim(len(boot) - 0.5, -0.5)
    axs[0].set_title("(a) Macro-F1", loc="left", fontweight="bold", fontsize=12)
    axs[1].set_title("(b) MCC", loc="left", fontweight="bold", fontsize=12)
    fig.text(0.5, -0.02, f"Points: paired block-bootstrap median; bars: 95% interval ({N_BOOT:,} resamples of the test blocks). "
             "Negative values favour the proposed model.", ha="center", va="top", fontsize=11)
    fig.tight_layout()
    _save(fig, stem)


def fig_band_attention(mean, sd, band_group, stem="Fig_band_attention", top=None):
    """Mean band-attention weight per input band over the test pixels (± SD over seeds), coloured by feature group."""
    order = mean.sort_values(ascending=False).index[:top] if top else mean.sort_values(ascending=False).index
    groups = list(dict.fromkeys(band_group[b] for b in order))
    pal = dict(zip(groups, plt.get_cmap("tab10").colors))
    fig, ax = plt.subplots(figsize=(7.0, 0.17 * len(order) + 1.2))
    y = np.arange(len(order))
    ax.barh(y, mean[order], xerr=sd[order] if sd is not None and sd.notna().any() else None,
            color=[pal[band_group[b]] for b in order], edgecolor="black", linewidth=0.4, error_kw={"lw": 0.6}, zorder=2)
    ax.set_yticks(y); ax.set_yticklabels(order, fontsize=11); ax.set_ylim(len(order) - 0.5, -0.5)
    ax.set_xlabel("Mean band-attention weight (test pixels)"); ax.grid(axis="x", color=GRID, lw=0.6); ax.set_axisbelow(True)
    ax.legend(handles=[Patch(facecolor=pal[g], edgecolor="black", linewidth=0.4, label=g) for g in groups],
              loc="lower right", frameon=True, edgecolor="black", fontsize=11)
    ax.set_title("Band attention of the pixel branch", fontsize=12, fontweight="bold")
    fig.tight_layout()
    _save(fig, stem)


# ---------------------------------------------------------------- maps
def fig_split():
    fig, axs = plt.subplots(1, 2, figsize=(10, 4.6))
    code = np.full((ROWS, COLS), np.nan)
    for i, m in enumerate((train_mask, buffer_mask, val_mask, test_mask)):
        code[m] = i
    axs[0].imshow(np.ma.masked_invalid(code), cmap=ListedColormap(SPLIT_COLORS), vmin=0, vmax=3, interpolation="nearest")
    axs[0].legend(handles=[Patch(facecolor=c, edgecolor="black", linewidth=0.4, label=l) for c, l in zip(SPLIT_COLORS, SPLIT_LABELS)],
                  loc="upper center", bbox_to_anchor=(0.5, -0.02), ncol=4, frameon=False)
    ref = np.full((ROWS, COLS), 255, np.uint8); ref[valid] = Y[valid]
    show_cls(axs[1], ref)
    axs[1].legend(handles=cls_handles(), loc="upper center", bbox_to_anchor=(0.5, -0.02), ncol=2, frameon=False)
    for ax, t in zip(axs, (f"(a) Spatial block split ({BLOCK_SIZE}-pixel blocks, {BUFFER_PX}-pixel buffer)",
                           "(b) Reference HI classes")):
        outline(ax); _map_axes(ax, t)
    fig.tight_layout()
    _save(fig, "Fig_split")


def fig_model_maps(name, prob, stem=None):
    """Probability, class map, error map on the test blocks and error map over the whole area."""
    cls, err_test = class_and_error_maps(prob, scope=test_mask)
    _, err_all = class_and_error_maps(prob)
    fig, axs = plt.subplots(1, 4, figsize=(17, 4.2 * ROWS / COLS + 1.4))
    im = show_prob(axs[0], prob)
    fig.colorbar(im, ax=axs[0], orientation="horizontal", fraction=0.05, pad=0.03, aspect=30, label="P(higher HI)")
    show_cls(axs[1], cls); show_err(axs[2], err_test); show_err(axs[3], err_all)
    axs[1].legend(handles=cls_handles(), loc="upper center", bbox_to_anchor=(0.5, -0.02), ncol=1, frameon=False)
    axs[2].legend(handles=err_handles(err_test), loc="upper center", bbox_to_anchor=(0.5, -0.02), ncol=3, frameon=False, fontsize=11)
    axs[3].legend(handles=err_handles(err_all), loc="upper center", bbox_to_anchor=(0.5, -0.02), ncol=3, frameon=False, fontsize=11)
    for ax, t in zip(axs, ("(a) Probability", f"(b) Classes (threshold {THRESHOLD})", "(c) Errors, test blocks",
                           "(d) Errors, whole area")):
        outline(ax); _map_axes(ax, t)
    fig.tight_layout(rect=(0, 0, 1, 0.95))
    fig.suptitle(name, fontweight="bold", fontsize=11, y=0.97)
    _save(fig, stem or f"Fig_maps_{slug(name)}")


def fig_overview(probs, kind="probability", ncols=4, stem=None):
    """All models side by side: probability maps or whole-area error maps (test blocks outlined)."""
    names = list(probs); nrows = int(np.ceil(len(names) / ncols))
    fig, axs = plt.subplots(nrows, ncols, figsize=(4.6 * ncols, (4.4 * ROWS / COLS + 0.35) * nrows + 0.6), squeeze=False)
    for ax, n in zip(axs.flat, names):
        if kind == "probability":
            im = show_prob(ax, probs[n])
        else:
            show_err(ax, class_and_error_maps(probs[n])[1])
        outline(ax); _map_axes(ax, n)
    for ax in list(axs.flat)[len(names):]:
        ax.axis("off")
    fig.tight_layout(rect=(0, 0.6 / fig.get_figheight(), 1, 1))
    if kind == "probability":
        cax = fig.add_axes([0.35, 0.25 / fig.get_figheight(), 0.3, 0.12 / fig.get_figheight()])
        fig.colorbar(im, cax=cax, orientation="horizontal", label="P(higher HI)")
    else:
        fig.legend(handles=err_handles(), loc="lower center", ncol=5, frameon=False, bbox_to_anchor=(0.5, 0.0))
    _save(fig, stem or f"Fig_overview_{kind}")


def fig_learning_curves(histories, stem="Fig_learning_curves"):
    names = list(histories); ncols = 4; nrows = int(np.ceil(len(names) / ncols))
    fig, axs = plt.subplots(nrows, ncols, figsize=(3.2 * ncols, 2.5 * nrows), squeeze=False)
    for ax, n in zip(axs.flat, names):
        h = histories[n]
        ax.plot(np.arange(1, len(h["loss"]) + 1), h["loss"], color=C1, lw=1.2, label="Training")
        ax.plot(np.arange(1, len(h["val_loss"]) + 1), h["val_loss"], color=C0, lw=1.2, label="Validation")
        ax.axvline(int(np.argmin(h["val_loss"])) + 1, color="black", ls=":", lw=0.8)
        ax.set_title(n, fontsize=10, fontweight="bold"); ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
        ax.grid(color=GRID, lw=0.6)
    for ax in list(axs.flat)[len(names):]:
        ax.axis("off")
    fig.legend(handles=[Line2D([], [], color=C1, label="Training"), Line2D([], [], color=C0, label="Validation"),
                        Line2D([], [], color="black", ls=":", label="Best validation epoch")],
               loc="lower center", ncol=3, frameon=False, bbox_to_anchor=(0.5, -0.03))
    fig.tight_layout(rect=(0, 0.04, 1, 1))
    _save(fig, stem)

In [ ]:
fig_split()

## 6. Models
### 6.1 Data pipelines
- **Center-pixel models** (HazardFuse, Patch CNN): a 21 × 21 patch centered on each labelled pixel; the label is that of the center pixel. Reflect padding at the raster edge.
- **Pixel models** (Random Forest, XGBoost): the band vector of each training pixel.
- **Segmentation models**: 32 × 32 windows (stride 8) with at least 25% labelled pixels of the subset. The loss (class-weighted BCE + Dice) is computed only on pixels of that subset, so neighbouring blocks provide context but never labels. Inference blends overlapping windows with a Gaussian weight and averages the 8 dihedral transforms (`SEG_TTA`).
- **Augmentation** (training only): random 90° rotations and flips.

In [ ]:
# ---- center-pixel patch models (HazardFuse, Patch CNN)
def padded(X, patch=PATCH):
    """Reflect-padded copy of a (rows, cols, bands) stack as a TensorFlow constant."""
    p = patch // 2
    return tf.constant(np.pad(X, ((p, p), (p, p), (0, 0)), mode="reflect"))


def augment_patch(x):
    x = tf.image.rot90(x, k=tf.random.uniform([], 0, 4, dtype=tf.int32))
    return tf.image.random_flip_up_down(tf.image.random_flip_left_right(x))


def patch_dataset(Xp, mask, training, seed=BASE_SEED, labels=None, patch=PATCH, batch=PATCH_BATCH):
    """(patch, label, class weight) for every pixel in `mask`; the patch is centered on the pixel."""
    labels = Y if labels is None else labels
    r, c = np.nonzero(mask)
    y = labels[r, c].astype(np.float32)
    w = np.where(y == 1, W_POS, W_NEG).astype(np.float32)
    bands = int(Xp.shape[-1])
    ds = tf.data.Dataset.from_tensor_slices((r.astype(np.int32), c.astype(np.int32), y, w))
    if training:
        ds = ds.shuffle(len(y), seed=seed, reshuffle_each_iteration=True)

    def _map(ri, ci, yi, wi):
        x = tf.slice(Xp, [ri, ci, 0], [patch, patch, bands])
        return (augment_patch(x) if training else x), yi, wi

    return ds.map(_map, num_parallel_calls=AUTOTUNE).batch(batch).prefetch(AUTOTUNE)


def patch_batches(Xp, mask, patch=PATCH, batch=4096):
    """Unlabelled patches for every pixel in `mask` (inference), and the pixel coordinates."""
    r, c = np.nonzero(mask)
    bands = int(Xp.shape[-1])
    ds = (tf.data.Dataset.from_tensor_slices((r.astype(np.int32), c.astype(np.int32)))
          .map(lambda ri, ci: tf.slice(Xp, [ri, ci, 0], [patch, patch, bands]), num_parallel_calls=AUTOTUNE)
          .batch(batch).prefetch(AUTOTUNE))
    return ds, (r, c)


def predict_patch(model, Xp, mask, patch=PATCH):
    """Probability map; NaN outside `mask`."""
    ds, (r, c) = patch_batches(Xp, mask, patch)
    prob = np.full(mask.shape, np.nan, np.float32)
    prob[r, c] = model.predict(ds, verbose=0).ravel()
    return prob


# ---- segmentation models (U-Net family, DeepLabv3+, SegFormer)
def window_starts(length, patch=SEG_PATCH, stride=SEG_STRIDE):
    if length < patch:
        raise ValueError(f"Raster side {length} is smaller than SEG_PATCH {patch}.")
    starts = list(range(0, length - patch + 1, stride))
    return starts if starts[-1] == length - patch else starts + [length - patch]   # full coverage


def seg_arrays(X, weight_mask, labels=None):
    """Patches whose labelled share (weight_mask) is at least MIN_LABEL_FRAC. Target channels: label, weight."""
    labels = Y if labels is None else labels
    xs, ys = [], []
    for r in window_starts(X.shape[0]):
        for c in window_starts(X.shape[1]):
            w = weight_mask[r:r + SEG_PATCH, c:c + SEG_PATCH].astype(np.float32)
            if w.mean() >= MIN_LABEL_FRAC:
                xs.append(X[r:r + SEG_PATCH, c:c + SEG_PATCH])
                ys.append(np.stack([labels[r:r + SEG_PATCH, c:c + SEG_PATCH], w], axis=-1))
    if not xs:
        raise ValueError("No patch meets MIN_LABEL_FRAC; lower it or increase BLOCK_SIZE.")
    return np.stack(xs).astype(np.float32), np.stack(ys).astype(np.float32)


def make_masked_bce_dice(w_pos, w_neg, dice_weight=1.0, eps=1e-7):
    """Class-weighted BCE + soft Dice, computed only where the weight channel is 1."""
    def masked_bce_dice(y_true, y_pred):
        y, m = y_true[..., 0:1], y_true[..., 1:2]
        p = tf.clip_by_value(y_pred, eps, 1.0 - eps)
        cw = y * w_pos + (1.0 - y) * w_neg
        bce = -(y * tf.math.log(p) + (1.0 - y) * tf.math.log(1.0 - p))
        bce = tf.reduce_sum(bce * m * cw) / (tf.reduce_sum(m * cw) + eps)
        dice = 1.0 - (2.0 * tf.reduce_sum(y * p * m) + 1.0) / (tf.reduce_sum(y * m) + tf.reduce_sum(p * m) + 1.0)
        return bce + dice_weight * dice
    return masked_bce_dice



def gaussian_window(size=SEG_PATCH, sigma_frac=0.25):
    g = np.exp(-0.5 * ((np.arange(size) - (size - 1) / 2) / (sigma_frac * size)) ** 2)
    return np.outer(g, g).astype(np.float32)


def predict_seg(model, X, mask, tta=SEG_TTA, batch=64):
    """Sliding-window inference with Gaussian-weighted blending; optional 8-fold dihedral test-time augmentation."""
    weight = gaussian_window()
    prob_sum, weight_sum = np.zeros(X.shape[:2], np.float32), np.zeros(X.shape[:2], np.float32)
    coords = [(r, c) for r in window_starts(X.shape[0]) for c in window_starts(X.shape[1])]
    n_t = 8 if tta else 1
    for i in range(0, len(coords), batch):
        chunk = coords[i:i + batch]
        xb = np.stack([X[r:r + SEG_PATCH, c:c + SEG_PATCH] for r, c in chunk])
        acc = np.zeros(xb.shape[:3], np.float32)
        for t in range(n_t):
            k, flip = t % 4, t >= 4
            xt = np.rot90(xb, k=k, axes=(1, 2))
            xt = xt[:, :, ::-1] if flip else xt
            pt = np.asarray(model.predict_on_batch(np.ascontiguousarray(xt)))[..., 0]
            acc += np.rot90(pt[:, :, ::-1] if flip else pt, k=-k, axes=(1, 2))
        for (r, c), p in zip(chunk, acc / n_t):
            prob_sum[r:r + SEG_PATCH, c:c + SEG_PATCH] += p * weight
            weight_sum[r:r + SEG_PATCH, c:c + SEG_PATCH] += weight
    prob = np.divide(prob_sum, weight_sum, out=np.full(X.shape[:2], np.nan, np.float32), where=weight_sum > 0)
    prob[~mask] = np.nan
    return np.clip(prob, 0.0, 1.0)

### 6.2 HazardFuse

| Component | Layers | Role |
|---|---|---|
| Center-pixel band-attention branch | center-pixel bands → band attention (Dense *B*/2, ReLU → Dense *B*, sigmoid) → [raw, attended] → Dense 96 → dropout | spectral signature of the pixel, with learned band weights |
| Patch spatial-embedding branch | Conv 3 × 3 (32) → BN → ReLU → Conv 3 × 3 (64) → BN → ReLU → max-pool → separable conv (96) → global average pooling → dropout | spatial context of the 21 × 21 neighbourhood |
| Gated fusion | concatenation → Dense 96 (ReLU) × sigmoid gate (Dense 96) → dropout → sigmoid | weights the joint embedding by a gate learned from both branches |

The switches in `HAZARDFUSE` (configuration) remove one component each; they are used by the architecture ablation.

In [ ]:
def build_hazardfuse(input_shape, dropout=0.30, band_attention=True, center_branch=True, spatial_branch=True,
                     fusion_gate=True, name="HazardFuse"):
    """
    HazardFuse: gated fusion of a center-pixel band-attention branch and a patch spatial-embedding branch.

    Center-pixel band-attention branch : bands of the center pixel, re-weighted by a band-attention gate
        (Dense B/2 ReLU -> Dense B sigmoid); the raw and attended vectors are concatenated and embedded (Dense 96).
    Patch spatial-embedding branch : Conv 3x3 (32) - BN - ReLU, Conv 3x3 (64) - BN - ReLU, max-pool,
        separable conv (96), global average pooling.
    Fusion       : concatenation -> Dense 96, multiplied element-wise by a sigmoid fusion gate learned from the
                   same concatenation -> dropout -> sigmoid output.
    The keyword switches remove one component each (architecture ablation).
    """
    h, w, bands = input_shape
    if h % 2 == 0 or w % 2 == 0:
        raise ValueError("HazardFuse needs an odd patch size (a defined center pixel).")
    if not (center_branch or spatial_branch):
        raise ValueError("At least one branch must be kept.")
    inp = layers.Input(input_shape, name="patch")
    branches = []

    if center_branch:                                       # center-pixel band-attention branch
        center = layers.Reshape((bands,), name="center_pixel")(
            layers.Cropping2D(((h // 2, h // 2), (w // 2, w // 2)), name="center_crop")(inp))
        if band_attention:
            a = layers.Dense(max(16, bands // 2), activation="relu", name="band_attention_hidden")(center)
            a = layers.Dense(bands, activation="sigmoid", name="band_attention")(a)
            px = layers.Concatenate(name="raw_and_attended")([center, layers.Multiply(name="attended_bands")([center, a])])
        else:
            px = center
        px = layers.Dense(96, activation="relu", name="center_embedding")(px)
        branches.append(layers.Dropout(dropout, name="center_dropout")(px))

    if spatial_branch:                                      # patch spatial-embedding branch
        s = inp
        for i, f in enumerate((32, 64), start=1):
            s = layers.Conv2D(f, 3, padding="same", use_bias=False, name=f"spatial_conv{i}")(s)
            s = layers.Activation("relu", name=f"spatial_relu{i}")(layers.BatchNormalization(name=f"spatial_bn{i}")(s))
        s = layers.MaxPooling2D(name="spatial_pool")(s)
        s = layers.SeparableConv2D(96, 3, padding="same", activation="relu", name="spatial_sepconv")(s)
        s = layers.GlobalAveragePooling2D(name="spatial_embedding")(s)
        branches.append(layers.Dropout(dropout, name="spatial_dropout")(s))

    fused = layers.Concatenate(name="fusion")(branches) if len(branches) == 2 else branches[0]
    joint = layers.Dense(96, activation="relu", name="joint_embedding")(fused)
    if fusion_gate and len(branches) == 2:
        gate = layers.Dense(96, activation="sigmoid", name="fusion_gate")(fused)
        joint = layers.Multiply(name="gated_fusion")([joint, gate])
    joint = layers.Dropout(dropout, name="joint_dropout")(joint)
    out = layers.Dense(1, activation="sigmoid", dtype="float32", name="p_higher_hi")(joint)
    return Model(inp, out, name=name)

### 6.3 Benchmarks
Random Forest (100 trees, balanced class weights) and XGBoost (300 trees, depth 8, learning rate 0.05, `scale_pos_weight` from the training class ratio) are fitted on pixels. All segmentation networks use encoder widths (32, 64, 128); Attention U-Net gates each skip connection with an additive attention gate. DeepLabv3+ is a compact version with atrous spatial pyramid pooling (dilation 2 and 4) and a full-resolution decoder; SegFormer is a compact three-stage version with spatially reduced attention and an all-MLP decoder. Every network is built once below to check its output shape before training.

In [ ]:
def conv_bn_relu(x, filters, k=3):
    x = layers.Conv2D(filters, k, padding="same", use_bias=False)(x)
    return layers.Activation("relu")(layers.BatchNormalization()(x))


def conv_block(x, filters):
    return conv_bn_relu(conv_bn_relu(x, filters), filters)


def residual_block(x, filters, se=False):
    shortcut = layers.BatchNormalization()(layers.Conv2D(filters, 1, padding="same", use_bias=False)(x))
    x = conv_bn_relu(x, filters)
    x = layers.BatchNormalization()(layers.Conv2D(filters, 3, padding="same", use_bias=False)(x))
    if se:                                                   # squeeze-and-excitation (ratio 8)
        s = layers.GlobalAveragePooling2D(keepdims=True)(x)
        s = layers.Conv2D(max(filters // 8, 4), 1, activation="relu")(s)
        x = layers.Multiply()([x, layers.Conv2D(filters, 1, activation="sigmoid")(s)])
    return layers.Activation("relu")(layers.Add()([x, shortcut]))


def se_residual_block(x, filters):
    return residual_block(x, filters, se=True)


def build_patch_cnn(input_shape, name="Patch_CNN"):
    inp = layers.Input(input_shape)
    x = layers.MaxPooling2D()(conv_block(inp, 32))
    x = layers.MaxPooling2D()(conv_block(x, 64))
    x = layers.GlobalAveragePooling2D()(conv_bn_relu(x, 128))
    x = layers.Dropout(0.3)(layers.Dense(64, activation="relu")(layers.Dropout(0.4)(x)))
    return Model(inp, layers.Dense(1, activation="sigmoid", dtype="float32")(x), name=name)


def build_seg_net(input_shape, block_fn, filters=SEG_FILTERS, dropout=0.3, name="seg_net"):
    """U-Net encoder-decoder; block_fn sets the variant (plain, residual, SE-residual)."""
    inp = layers.Input(input_shape)
    x, skips = inp, []
    for f in filters[:-1]:
        x = block_fn(x, f); skips.append(x); x = layers.MaxPooling2D()(x)
    x = layers.SpatialDropout2D(dropout)(block_fn(x, filters[-1]))
    for f, skip in zip(reversed(filters[:-1]), reversed(skips)):
        x = block_fn(layers.Concatenate()([layers.UpSampling2D()(x), skip]), f)
    return Model(inp, layers.Conv2D(1, 1, activation="sigmoid", dtype="float32")(x), name=name)


def attention_gate(skip, gating, inter):
    """Additive attention gate (Oktay et al., 2018): the decoder map decides which skip features pass."""
    theta = layers.Conv2D(inter, 1, padding="same", use_bias=False)(skip)
    phi = layers.Conv2D(inter, 1, padding="same")(gating)
    psi = layers.Conv2D(1, 1, padding="same", activation="sigmoid")(layers.Activation("relu")(layers.Add()([theta, phi])))
    return layers.Multiply()([skip, psi])


def build_attention_unet(input_shape, filters=SEG_FILTERS, dropout=0.2, name="Attention_UNet"):
    inp = layers.Input(input_shape)
    x, skips = inp, []
    for f in filters[:-1]:
        x = conv_block(x, f); skips.append(x); x = layers.MaxPooling2D()(x)
    x = layers.SpatialDropout2D(dropout)(conv_block(x, filters[-1]))
    for f, skip in zip(reversed(filters[:-1]), reversed(skips)):
        up = layers.UpSampling2D()(x)
        x = conv_block(layers.Concatenate()([up, attention_gate(skip, up, max(f // 2, 8))]), f)
    return Model(inp, layers.Conv2D(1, 1, activation="sigmoid", dtype="float32")(x), name=name)


def build_unet_plusplus(input_shape, filters=SEG_FILTERS, dropout=0.2, name="UNet_plusplus"):
    """Nested U-Net; node (i, j) is at depth i after j refinements."""
    depth, inp, node = len(filters), layers.Input(input_shape), {}
    for i in range(depth):
        node[(i, 0)] = conv_block(inp if i == 0 else layers.MaxPooling2D()(node[(i - 1, 0)]), filters[i])
    node[(depth - 1, 0)] = layers.SpatialDropout2D(dropout)(node[(depth - 1, 0)])
    for j in range(1, depth):
        for i in range(depth - j):
            up = layers.UpSampling2D()(node[(i + 1, j - 1)])
            node[(i, j)] = conv_block(layers.Concatenate()([node[(i, k)] for k in range(j)] + [up]), filters[i])
    return Model(inp, layers.Conv2D(1, 1, activation="sigmoid", dtype="float32")(node[(0, depth - 1)]), name=name)


def aspp(x, filters=128, rates=(2, 4)):
    """Atrous spatial pyramid pooling; dilation rates scaled to a 32-pixel patch."""
    branches = [layers.Conv2D(filters, 1, padding="same", use_bias=False)(x)]
    branches += [layers.SeparableConv2D(filters, 3, padding="same", dilation_rate=r, use_bias=False)(x) for r in rates]
    pooled = conv_bn_relu(layers.GlobalAveragePooling2D(keepdims=True)(x), filters, k=1)
    branches.append(layers.UpSampling2D(size=(x.shape[1], x.shape[2]), interpolation="bilinear")(pooled))
    x = layers.Activation("relu")(layers.BatchNormalization()(layers.Concatenate()(branches)))
    return layers.Conv2D(filters, 1, padding="same", use_bias=False)(x)


def build_deeplabv3plus(input_shape, filters=SEG_FILTERS, dropout=0.2, name="DeepLabv3plus"):
    inp = layers.Input(input_shape)
    low = conv_block(inp, filters[0])                                          # full-resolution detail
    x = conv_block(layers.MaxPooling2D()(low), filters[1])
    x = conv_block(layers.MaxPooling2D()(x), filters[-1])                      # 1/4 resolution
    x = layers.SpatialDropout2D(dropout)(aspp(x, filters[-1]))
    x = layers.UpSampling2D(size=4, interpolation="bilinear")(x)
    x = conv_block(layers.Concatenate()([x, conv_bn_relu(low, 48, k=1)]), filters[1])
    return Model(inp, layers.Conv2D(1, 1, activation="sigmoid", dtype="float32")(x), name=name)


def _transformer_stage(x, dim, heads, reduction, mlp_ratio=2):
    """Efficient self-attention (spatially reduced keys and values) followed by an MLP."""
    h, w = x.shape[1], x.shape[2]
    tokens = layers.Reshape((h * w, dim))(x)
    y = layers.LayerNormalization(epsilon=1e-5)(tokens)
    if reduction > 1:
        kv = layers.Conv2D(dim, reduction, strides=reduction, padding="same")(layers.Reshape((h, w, dim))(y))
        kv = layers.LayerNormalization(epsilon=1e-5)(layers.Reshape(((h // reduction) * (w // reduction), dim))(kv))
    else:
        kv = y
    tokens = layers.Add()([tokens, layers.MultiHeadAttention(num_heads=heads, key_dim=max(dim // heads, 8))(y, kv)])
    y = layers.Dense(dim)(layers.Dense(dim * mlp_ratio, activation="gelu")(layers.LayerNormalization(epsilon=1e-5)(tokens)))
    return layers.Reshape((h, w, dim))(layers.Add()([tokens, y]))


def build_segformer(input_shape, dims=SEG_FILTERS, heads=(1, 2, 4), reductions=(4, 2, 1), decoder_dim=128,
                    dropout=0.2, name="SegFormer"):
    """Compact SegFormer: overlapping patch embeddings, three transformer stages, all-MLP decoder."""
    inp = layers.Input(input_shape)
    x = layers.Activation("gelu")(layers.BatchNormalization()(layers.Conv2D(dims[0], 7, strides=2, padding="same")(inp)))
    feats = []
    for k, (dim, head, red) in enumerate(zip(dims, heads, reductions)):
        if k:
            x = layers.Activation("gelu")(layers.BatchNormalization()(layers.Conv2D(dim, 3, strides=2, padding="same")(x)))
        x = _transformer_stage(x, dim, head, red)
        feats.append(x)
    target = feats[0].shape[1]
    merged = []
    for f in feats:
        f = layers.Conv2D(decoder_dim, 1, padding="same")(f)
        merged.append(f if f.shape[1] == target else
                      layers.UpSampling2D(size=target // f.shape[1], interpolation="bilinear")(f))
    x = layers.Conv2D(decoder_dim, 1, padding="same", use_bias=False)(layers.Concatenate()(merged))
    x = layers.SpatialDropout2D(dropout)(layers.Activation("gelu")(layers.BatchNormalization()(x)))
    x = layers.UpSampling2D(size=input_shape[0] // target, interpolation="bilinear")(x)
    return Model(inp, layers.Conv2D(1, 1, activation="sigmoid", dtype="float32")(x), name=name)


# ---- registry: kind decides the data pipeline; family decides the marker in the comparison figures
def _forest_nodes(m):
    return sum(e.tree_.node_count for e in m.estimators_)


def _xgb_nodes(m):
    return sum(t.count("\n") for t in m.get_booster().get_dump())


MODELS = {
    PROPOSED:        dict(kind="patch", family="Proposed", build=lambda s: build_hazardfuse(s, **HAZARDFUSE)),
    "Random Forest": dict(kind="pixel", family="ML", size=_forest_nodes,
                          build=lambda seed: RandomForestClassifier(n_estimators=RF_TREES, n_jobs=-1, random_state=seed,
                                                                    class_weight="balanced_subsample" if USE_CLASS_WEIGHTS else None)),
    "XGBoost":       dict(kind="pixel", family="ML", size=_xgb_nodes,
                          build=lambda seed: XGBClassifier(**XGB_PARAMS, random_state=seed,
                                                           scale_pos_weight=(W_POS / W_NEG) if USE_CLASS_WEIGHTS else 1.0)),
    "Patch CNN":     dict(kind="patch", family="DL", build=build_patch_cnn),
    "U-Net":         dict(kind="seg", family="DL", build=lambda s: build_seg_net(s, conv_block, name="UNet")),
    "U-Net++":       dict(kind="seg", family="DL", build=build_unet_plusplus),
    "Attention U-Net": dict(kind="seg", family="DL", build=build_attention_unet),
    "ResU-Net":      dict(kind="seg", family="DL", build=lambda s: build_seg_net(s, residual_block, name="ResUNet")),
    "SE-ResU-Net":   dict(kind="seg", family="DL", build=lambda s: build_seg_net(s, se_residual_block, dropout=0.2,
                                                                                   name="SE_ResUNet")),
    "DeepLabv3+":    dict(kind="seg", family="DL", build=build_deeplabv3plus),
    "SegFormer":     dict(kind="seg", family="DL", build=build_segformer),
}
if XGBClassifier is None:
    MODELS.pop("XGBoost")

# every network must build and return the expected output shape before any training starts
_rows = []
for _n, _s in MODELS.items():
    if _s["kind"] == "pixel":
        continue
    _shape = (PATCH, PATCH, len(IN_IDX)) if _s["kind"] == "patch" else (SEG_PATCH, SEG_PATCH, len(IN_IDX))
    _m = _s["build"](_shape)
    _exp = (None, 1) if _s["kind"] == "patch" else (None, SEG_PATCH, SEG_PATCH, 1)
    assert tuple(_m.output_shape) == _exp, f"{_n}: output {_m.output_shape}, expected {_exp}"
    _rows.append({"Model": _n, "Input": str(_shape), "Output": str(_m.output_shape), "Parameters": _m.count_params()})
    del _m
tf.keras.backend.clear_session()
T_ARCH = pd.DataFrame(_rows).set_index("Model")
display(T_ARCH)

## 7. Training with seed repeats
Adam; early stopping (patience 20) and learning-rate halving on validation loss; the weights with the lowest validation loss are restored. Each fitted model is saved to `exported_models/fitted` (the Random Forest for the first seed only, because of its size).

In [ ]:
SEG_LOSS = make_masked_bce_dice(W_POS, W_NEG)            # class weights from the training pixels


def fit_keras(model, train_ds, val_ds, loss, lr, epochs, tag):
    """Adam; early stopping and learning-rate reduction on validation loss; best weights restored."""
    ckpt = CKPT_DIR / f"{slug(tag)}.weights.h5"
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr), loss=loss)
    callbacks = [EarlyStopping(monitor="val_loss", patience=PATIENCE, restore_best_weights=True),
                 ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=max(PATIENCE // 3, 2), min_lr=1e-6),
                 ModelCheckpoint(str(ckpt), monitor="val_loss", save_best_only=True, save_weights_only=True)]
    t0 = time.time()
    hist = model.fit(train_ds, validation_data=val_ds, epochs=epochs, callbacks=callbacks, verbose=0)
    seconds = time.time() - t0
    model.load_weights(str(ckpt))
    return {k: list(map(float, v)) for k, v in hist.history.items()}, seconds


def make_inputs(kind, X):
    """Data objects that do not depend on the seed, built once per input stack."""
    if kind == "patch":
        return {"Xp": padded(X)}
    if kind == "seg":
        return {"train": seg_arrays(X, train_mask), "val": seg_arrays(X, val_mask)}
    return {"X_train": X[train_mask], "y_train": Y[train_mask], "X_valid": X[valid]}


def _seg_ds(arrays, training, seed):
    xs, ys = arrays
    ds = tf.data.Dataset.from_tensor_slices((xs, ys))
    if training:
        b = xs.shape[-1]
        ds = ds.shuffle(len(xs), seed=seed, reshuffle_each_iteration=True).map(
            lambda x, y: tuple(tf.split(augment_patch(tf.concat([x, y], -1)), [b, 2], axis=-1)), num_parallel_calls=AUTOTUNE)
    return ds.batch(SEG_BATCH).prefetch(AUTOTUNE)


def run_once(name, seed, X, inputs, spec=None, tag=None, save_model=False):
    """Fit one model with one seed on stack X; return the probability map, metrics on each split and the cost."""
    spec = MODELS[name] if spec is None else spec
    tag = tag or f"{name} seed {seed}"
    set_seed(seed)
    kind, history = spec["kind"], None
    if kind == "pixel":
        model = spec["build"](seed)
        t0 = time.time(); model.fit(inputs["X_train"], inputs["y_train"]); seconds = time.time() - t0
        prob = np.full((ROWS, COLS), np.nan, np.float32)
        prob[valid] = model.predict_proba(inputs["X_valid"])[:, 1]
        n_params = spec["size"](model)                                                # tree nodes
    else:
        shape = (PATCH, PATCH, X.shape[-1]) if kind == "patch" else (SEG_PATCH, SEG_PATCH, X.shape[-1])
        model = spec["build"](shape)
        if kind == "patch":
            Xp = inputs["Xp"]
            history, seconds = fit_keras(model, patch_dataset(Xp, train_mask, True, seed), patch_dataset(Xp, val_mask, False),
                                         "binary_crossentropy", PATCH_LR, PATCH_EPOCHS, tag)
            prob = predict_patch(model, Xp, valid)
        else:
            history, seconds = fit_keras(model, _seg_ds(inputs["train"], True, seed), _seg_ds(inputs["val"], False, seed),
                                         SEG_LOSS, SEG_LR, SEG_EPOCHS, tag)
            prob = predict_seg(model, X, valid)
        n_params = model.count_params()
    rec = {"seed": seed, "prob": prob.astype(np.float32), "seconds": seconds, "params": n_params, "history": history,
           "metrics": {s: compute_metrics(Y[m], prob[m]) for s, m in MASKS.items()}}
    if save_model and (name != "Random Forest" or seed == BASE_SEED):  # one forest is enough on disk
        mdir = EXPORT_DIR / "fitted" / slug(name); mdir.mkdir(parents=True, exist_ok=True)
        if kind == "pixel":
            rec["model_file"] = str(mdir / f"seed{seed}.joblib"); joblib.dump(model, rec["model_file"], compress=3)
        else:
            rec["model_file"] = str(mdir / f"seed{seed}.keras"); model.save(rec["model_file"])
    del model; gc.collect()
    m = rec["metrics"]["test"]
    print(f"  {tag:<38s} test Macro-F1 {m['Macro_F1']:.4f} | MCC {m['MCC']:.4f} | kappa {m['Kappa']:.4f} | "
          f"ROC-AUC {m['ROC_AUC_1']:.4f} | {seconds:6.1f} s" + (f" | {len(history['loss'])} epochs" if history else ""))
    return rec


def run_repeats(name, X, seeds, spec=None, label=None, save_model=False):
    spec = MODELS[name] if spec is None else spec
    inputs = make_inputs(spec["kind"], X)
    runs = [run_once(name, s, X, inputs, spec, tag=f"{label or name} seed {s}", save_model=save_model) for s in seeds]
    del inputs; tf.keras.backend.clear_session(); gc.collect()
    return runs


def summarise(runs, split="test", cols=None):
    """Mean and SD over seeds of every metric for one model."""
    df = pd.DataFrame([r["metrics"][split] for r in runs])
    cols = cols or list(df.columns)
    return df[cols].mean(), df[cols].std(ddof=1) if len(df) > 1 else df[cols].std(ddof=0) * np.nan

In [ ]:
RUN_SEEDS = SEEDS if RUN_SEED_REPEATS else (BASE_SEED,)
assert BASE_SEED in RUN_SEEDS
RUNS = {}
for _name in MODELS:
    print(f"\n{_name} ({MODELS[_name]['kind']}), seeds {RUN_SEEDS}")
    RUNS[_name] = run_repeats(_name, X_IN, RUN_SEEDS, save_model=EXPORT_MODELS)


def main_run(name, runs=None):
    return next(r for r in (RUNS if runs is None else runs)[name] if r["seed"] == BASE_SEED)


PROB = {n: main_run(n)["prob"] for n in MODELS}                      # main-seed probability maps
FAMILY = {n: s["family"] for n, s in MODELS.items()}

# per-seed metrics, all splits (no maps) - the raw material of every table below
LONG = pd.DataFrame([{"Model": n, "Seed": r["seed"], "Split": s, "Seconds": r["seconds"], "Parameters": r["params"], **m}
                     for n, runs in RUNS.items() for r in runs for s, m in r["metrics"].items()])
LONG.to_csv(TAB_DIR / "per_seed_metrics.csv", index=False)
if SAVE_GEOTIFFS:
    for _n, _p in PROB.items():
        save_map_geotiffs(_n, _p, labels=Y)

## 8. Model comparison
### 8.1 Tables
T1: test metrics (mean ± SD over seeds), ranked by Macro-F1. T2: class-wise precision, recall and F1. T3: training, validation and test scores (overfitting check).

In [ ]:
KEY = ["Accuracy", "Balanced_Acc", "Macro_F1", "Kappa", "MCC", "IoU_1", "ROC_AUC_1", "PR_AUC_1", "Brier"]
CLASSWISE = ["Precision_0", "Recall_0", "F1_0", "Precision_1", "Recall_1", "F1_1"]


def pm(mu, sd, d=3):
    return f"{mu:.{d}f}" if not np.isfinite(sd) else f"{mu:.{d}f} ± {sd:.{d}f}"


test = LONG[LONG.Split == "test"].groupby("Model", sort=False)
MEAN, SD = test[KEY + CLASSWISE].mean(), test[KEY + CLASSWISE].std(ddof=1)
ORDER = MEAN["Macro_F1"].sort_values(ascending=False).index.tolist()          # rank by mean test Macro-F1
MEAN, SD = MEAN.loc[ORDER], SD.reindex(ORDER)
COST = test[["Parameters", "Seconds"]].mean().loc[ORDER]

T1 = pd.DataFrame({k: [pm(MEAN.at[m, k], SD.at[m, k]) for m in ORDER] for k in KEY}, index=ORDER)
T1["Parameters"] = [f"{int(v):,}" if FAMILY[m] != "ML" else f"{int(v):,} nodes" for m, v in COST["Parameters"].items()]
T1["Fit time (s)"] = COST["Seconds"].round(1).values
T1.index.name = f"Test blocks, mean ± SD over seeds {RUN_SEEDS}"
T1.to_csv(TAB_DIR / "T1_test_metrics.csv"); pd.concat({"mean": MEAN, "sd": SD}, axis=1).to_csv(TAB_DIR / "T1_test_metrics_numeric.csv")
display(T1)

T2 = MEAN[CLASSWISE].round(3)
T2.columns = pd.MultiIndex.from_tuples([(c, k) for c in ("Class 0 (lower HI)", "Class 1 (higher HI)")
                                        for k in ("Precision", "Recall", "F1")])
T2.to_csv(TAB_DIR / "T2_classwise.csv")
display(T2)

T3 = (LONG.groupby(["Model", "Split"])[["Macro_F1", "MCC"]].mean().unstack("Split")
      .reindex(columns=["train", "val", "test"], level=1).loc[ORDER].round(3))
T3[("Macro_F1", "train − test")] = (T3[("Macro_F1", "train")] - T3[("Macro_F1", "test")]).round(3)
T3.to_csv(TAB_DIR / "T3_train_val_test.csv")
display(T3)

### 8.2 Figures
Model comparison (overall and class-wise scores), ROC and precision–recall space, HazardFuse performance by split, HazardFuse maps (probability, classes, errors on the test blocks and over the whole area), all models side by side, and learning curves. In whole-area error maps most pixels were used for training; read them as spatial coverage, not as accuracy. Test blocks are outlined.

In [ ]:
fig_model_comparison(MEAN)
fig_roc_pr_space(MEAN, FAMILY)
fig_split_performance(RUNS[PROPOSED])
fig_model_maps(PROPOSED, PROB[PROPOSED])
fig_overview({n: PROB[n] for n in ORDER}, "probability")
fig_overview({n: PROB[n] for n in ORDER}, "error")
fig_learning_curves({n: main_run(n)["history"] for n in ORDER if main_run(n)["history"]})

### 8.3 Paired block bootstrap
The test blocks are resampled with replacement; the same resamples are used for every model, so each difference is paired. Intervals are 2.5th–97.5th percentiles (seed 42 models).

In [ ]:
test_blocks = np.unique(block_id[test_mask])
_bidx = np.searchsorted(test_blocks, block_id[test_mask])
_ytest = Y[test_mask].astype(int)


def block_counts(prob, threshold=THRESHOLD):
    """TN, FP, FN, TP per test block, shape (blocks, 4)."""
    pred = (prob[test_mask] >= threshold).astype(int)
    code = 2 * _ytest + pred                                   # 0 TN, 1 FP, 2 FN, 3 TP
    return np.stack([np.bincount(_bidx[code == k], minlength=len(test_blocks)) for k in range(4)], axis=1)


_rs = np.random.default_rng(BASE_SEED).integers(0, len(test_blocks), size=(N_BOOT, len(test_blocks)))
BOOT_MET = {n: metrics_from_counts(*block_counts(p)[_rs].sum(axis=1).T) for n, p in PROB.items()}   # same resamples for all


def _ci(v):
    return np.nanpercentile(v, [50, 2.5, 97.5])


rows = {}
for n in ORDER:
    r = {}
    for key, short in (("Macro_F1", "Macro_F1"), ("MCC", "MCC")):
        r[short], r[f"{short}_lo"], r[f"{short}_hi"] = _ci(BOOT_MET[n][key])
        if n != PROPOSED:
            d = BOOT_MET[n][key] - BOOT_MET[PROPOSED][key]
            r[f"d{short}"], r[f"d{short}_lo"], r[f"d{short}_hi"] = _ci(d)
            r[f"P(d{short} < 0)"] = np.mean(d < 0)
    rows[n] = r
BOOT = pd.DataFrame(rows).T
BOOT.to_csv(TAB_DIR / "T4_block_bootstrap.csv")
T4 = pd.DataFrame({"Macro-F1 (95% CI)": [f"{r.Macro_F1:.3f} ({r.Macro_F1_lo:.3f} to {r.Macro_F1_hi:.3f})" for r in BOOT.itertuples()],
                   "ΔMacro-F1 vs proposed (95% CI)": ["—" if n == PROPOSED else f"{BOOT.at[n, 'dMacro_F1']:+.3f} "
                                                      f"({BOOT.at[n, 'dMacro_F1_lo']:+.3f} to {BOOT.at[n, 'dMacro_F1_hi']:+.3f})"
                                                      for n in BOOT.index],
                   "ΔMCC vs proposed (95% CI)": ["—" if n == PROPOSED else f"{BOOT.at[n, 'dMCC']:+.3f} "
                                                 f"({BOOT.at[n, 'dMCC_lo']:+.3f} to {BOOT.at[n, 'dMCC_hi']:+.3f})"
                                                 for n in BOOT.index]}, index=BOOT.index)
T4.index.name = f"Seed {BASE_SEED}; {len(test_blocks)} test blocks, {N_BOOT:,} paired resamples"
display(T4)
fig_bootstrap(BOOT.drop(index=PROPOSED))

## 9. Band attention
Mean output of the band-attention gate over the test pixels, for each seed. A high weight means the center-pixel branch passes that band on with little attenuation; it is not a causal importance.

In [ ]:
def band_attention_weights(model_file, X, mask):
    """Mean sigmoid band-attention weight per input band over the pixels in `mask`."""
    m = tf.keras.models.load_model(model_file, compile=False)
    sub = Model(m.input, m.get_layer("band_attention").output)
    ds, _ = patch_batches(padded(X), mask)
    return sub.predict(ds, verbose=0).mean(axis=0)


if HAZARDFUSE["band_attention"] and HAZARDFUSE["center_branch"] and EXPORT_MODELS:
    _W = pd.DataFrame({r["seed"]: band_attention_weights(r["model_file"], X_IN, test_mask) for r in RUNS[PROPOSED]},
                      index=IN_NAMES)
    _W.to_csv(TAB_DIR / "T5_band_attention.csv")
    BAND_GROUP = {RAW_NAMES[i]: g for g, idx in GROUP_IDX.items() for i in idx}
    fig_band_attention(_W.mean(axis=1), _W.std(axis=1, ddof=1) if _W.shape[1] > 1 else None,
                       {b: BAND_GROUP.get(b, "Other") for b in IN_NAMES})
    display(_W.mean(axis=1).sort_values(ascending=False).head(15).round(3).to_frame("Mean weight"))

## 10. Ablations
### 10.1 Architecture
Each variant removes one component of HazardFuse and is refitted with the same seeds; differences are paired by seed (variant − full). Removing a branch also removes the fusion gate, and removing the center-pixel branch removes band attention.

In [ ]:
ARCH_VARIANTS = {                     # each removes one component; the full model is the benchmark run above
    "w/o band attention":           dict(band_attention=False),
    "w/o center-pixel branch":      dict(center_branch=False, band_attention=False, fusion_gate=False),
    "w/o fusion gate":              dict(fusion_gate=False),
    "w/o spatial-embedding branch": dict(spatial_branch=False, fusion_gate=False),
}
ABL_METRICS = ["Accuracy", "Balanced_Acc", "Macro_F1", "Kappa", "MCC", "IoU_1", "ROC_AUC_1", "PR_AUC_1", "Recall_0", "Recall_1"]


def paired_deltas(variant_runs, full_runs, metrics=ABL_METRICS, split="test"):
    """Per-seed differences variant − full (same seed), then mean and SD over seeds."""
    full = {r["seed"]: r["metrics"][split] for r in full_runs}
    d = pd.DataFrame([{k: r["metrics"][split][k] - full[r["seed"]][k] for k in metrics}
                      for r in variant_runs if r["seed"] in full])
    return d.mean(), d.std(ddof=1) if len(d) > 1 else d.std(ddof=0) * np.nan


if RUN_ARCH_ABLATION:
    full_runs = [r for r in RUNS[PROPOSED] if r["seed"] in ABLATION_SEEDS]
    ARCH_RUNS, ARCH_PARAMS = {}, {"Full": full_runs[0]["params"]}
    for v, switches in ARCH_VARIANTS.items():
        spec = dict(MODELS[PROPOSED], build=lambda s, sw={**HAZARDFUSE, **switches}: build_hazardfuse(s, **sw))
        print(f"\n{v}")
        ARCH_RUNS[v] = run_repeats(PROPOSED, X_IN, ABLATION_SEEDS, spec=spec, label=f"HazardFuse {v}")
        ARCH_PARAMS[v] = ARCH_RUNS[v][0]["params"]
    _d = {v: paired_deltas(runs, full_runs) for v, runs in ARCH_RUNS.items()}
    ARCH_DELTA = pd.DataFrame({v: d[0] for v, d in _d.items()}).T
    _fm, _fs = summarise(full_runs, cols=ABL_METRICS)
    T6 = pd.DataFrame({"Macro-F1": [pm(_fm["Macro_F1"], _fs["Macro_F1"])] +
                                   [pm(*[x["Macro_F1"] for x in summarise(ARCH_RUNS[v], cols=ABL_METRICS)]) for v in ARCH_RUNS],
                       "ΔMacro-F1": ["—"] + [pm(_d[v][0]["Macro_F1"], _d[v][1]["Macro_F1"]) for v in ARCH_RUNS],
                       "ΔIoU (class 1)": ["—"] + [pm(_d[v][0]["IoU_1"], _d[v][1]["IoU_1"]) for v in ARCH_RUNS],
                       "ΔROC-AUC": ["—"] + [pm(_d[v][0]["ROC_AUC_1"], _d[v][1]["ROC_AUC_1"]) for v in ARCH_RUNS],
                       "Parameters": [f"{ARCH_PARAMS[k]:,}" for k in ["Full"] + list(ARCH_RUNS)]},
                      index=[f"{PROPOSED} (full)"] + list(ARCH_RUNS))
    T6.index.name = f"Test blocks; Δ = variant − full, paired by seed, mean ± SD over seeds {ABLATION_SEEDS}"
    T6.to_csv(TAB_DIR / "T6_architecture_ablation.csv"); ARCH_DELTA.to_csv(TAB_DIR / "T6_architecture_ablation_deltas.csv")
    display(T6)
    fig_arch_ablation(ARCH_DELTA, {v: ARCH_PARAMS[v] for v in ARCH_RUNS}, ARCH_PARAMS["Full"])

### 10.2 Feature groups
Reference: all bands except QA60, i.e. the benchmark input; its runs are reused from section 7. Each configuration removes one group and is refitted with the same seeds. The shaded band is ± 1 SD of the reference Macro-F1 over seeds; changes inside it are within seed-to-seed variation.

In [ ]:
# Feature-group ablation of HazardFuse. Reference = every band except QA60 (the benchmark input), so the
# "Full" runs are the HazardFuse runs of section 7 and are reused.
ABL_BASE = [i for i, n in enumerate(RAW_NAMES) if n.lower() != "qa60"]
ABL_GROUPS = {g: idx for g, idx in GROUP_IDX.items() if g != "Quality" and any(i in ABL_BASE for i in idx)}

if RUN_FEATURE_ABLATION:
    configs = {"Full": ABL_BASE}
    configs.update({f"w/o {g} ({len(idx)})": [i for i in ABL_BASE if i not in idx] for g, idx in ABL_GROUPS.items()})
    FEAT_RUNS = {}
    for cfg, idx in configs.items():
        if sorted(idx) == sorted(IN_IDX) and set(ABLATION_SEEDS) <= set(RUN_SEEDS):
            FEAT_RUNS[cfg] = [r for r in RUNS[PROPOSED] if r["seed"] in ABLATION_SEEDS]     # identical run, reused
            print(f"\n{cfg}: same input as the benchmark run - reused")
            continue
        print(f"\n{cfg}: {len(idx)} bands")
        FEAT_RUNS[cfg] = run_repeats(PROPOSED, X_ALL[..., idx], ABLATION_SEEDS, label=f"HazardFuse {cfg}")

    full = FEAT_RUNS["Full"]
    _d = {c: paired_deltas(r, full) for c, r in FEAT_RUNS.items() if c != "Full"}
    FEAT_DELTA = pd.DataFrame({c: d[0] for c, d in _d.items()}).T
    FEAT_RECALL = pd.DataFrame({c: summarise(r, cols=["Recall_0", "Recall_1"])[0] for c, r in FEAT_RUNS.items()}).T
    _fm, _fs = summarise(full, cols=ABL_METRICS)
    T7 = pd.DataFrame({"Bands": [len(configs[c]) for c in FEAT_RUNS],
                       "Macro-F1": [pm(*[x["Macro_F1"] for x in summarise(FEAT_RUNS[c], cols=ABL_METRICS)]) for c in FEAT_RUNS],
                       "ΔMacro-F1": ["—"] + [pm(_d[c][0]["Macro_F1"], _d[c][1]["Macro_F1"]) for c in _d],
                       "ΔKappa": ["—"] + [pm(_d[c][0]["Kappa"], _d[c][1]["Kappa"]) for c in _d],
                       "ΔIoU (class 1)": ["—"] + [pm(_d[c][0]["IoU_1"], _d[c][1]["IoU_1"]) for c in _d]},
                      index=list(FEAT_RUNS))
    T7.index.name = f"Test blocks; Δ = variant − full, paired by seed, mean ± SD over seeds {ABLATION_SEEDS}"
    T7.to_csv(TAB_DIR / "T7_feature_ablation.csv"); FEAT_DELTA.to_csv(TAB_DIR / "T7_feature_ablation_deltas.csv")
    display(T7)
    fig_feature_ablation(FEAT_DELTA, FEAT_RECALL.drop(index="Full"), FEAT_RECALL.loc["Full"].values,
                         noise=None if np.isnan(_fs["Macro_F1"]) else float(_fs["Macro_F1"]))

## 11. Export and transfer to a new site
Each bundle folder holds one model file per seed and `manifest.json` (band names and order, training mean and SD, patch sizes, threshold, training class share, test scores). To use a bundle in a new session without the Glen Lyn data, run the cells of sections 1 and 2, the two function cells of section 5, the cell of 6.1 and the second cell below (transfer functions), then call `transfer_to_site(bundle_dir, feature_tif, label_tif=None, site=...)`.

Transfer options:
- **Standardisation.** `source` uses the Glen Lyn statistics; `target` uses the new site's own statistics (no labels needed).
- **AdaBN.** Re-estimates the BatchNorm statistics on the new site (no labels needed).
- **Threshold.** `source` (0.5), `prevalence` (same predicted class-1 share as the Glen Lyn training labels), or a number.

The prediction is the mean over the seed members of the bundle.

In [ ]:
import shutil


def export_bundle(name):
    """Self-contained folder: one model file per seed + manifest.json with everything needed to re-apply it."""
    spec, runs = MODELS[name], RUNS[name]
    bdir = EXPORT_DIR / slug(name)
    bdir.mkdir(parents=True, exist_ok=True)
    files = []
    for r in runs:
        if r.get("model_file"):
            dst = bdir / f"{slug(name)}_seed{r['seed']}{Path(r['model_file']).suffix}"
            shutil.copy2(r["model_file"], dst); files.append(dst.name)
    manifest = {
        "model": name, "kind": spec["kind"], "family": spec["family"],
        "architecture": dict(HAZARDFUSE) if name == PROPOSED else None,
        "training_site": SITE, "created": datetime.now().isoformat(timespec="seconds"),
        "tensorflow": tf.__version__, "seeds": [r["seed"] for r in runs if r.get("model_file")], "files": files,
        "bands": IN_NAMES, "band_mean": BAND_MEAN[IN_IDX].tolist(), "band_std": BAND_STD[IN_IDX].tolist(),
        "nan_fill": "0 after standardisation (the training mean)",
        "patch": PATCH, "seg_patch": SEG_PATCH, "seg_stride": SEG_STRIDE, "seg_tta": SEG_TTA,
        "threshold": THRESHOLD, "train_class1_fraction": float(Y[train_mask].mean()),
        "class_weights": [W_NEG, W_POS], "split": dict(block=BLOCK_SIZE, fractions=SPLIT_FRAC, buffer=BUFFER_PX),
        "test_metrics_mean_over_seeds": {k: float(MEAN.at[name, k]) for k in KEY},
    }
    (bdir / "manifest.json").write_text(json.dumps(manifest, indent=2))
    return bdir


if EXPORT_MODELS:
    BUNDLES = {n: export_bundle(n) for n in MODELS}
    for n, d in BUNDLES.items():
        print(f"{n:<14s} -> {d}  ({len(json.loads((d / 'manifest.json').read_text())['files'])} model file(s))")

In [ ]:
def load_bundle(bundle_dir, seeds=None):
    bundle_dir = Path(bundle_dir)
    man = json.loads((bundle_dir / "manifest.json").read_text())
    pairs = [(s, f) for s, f in zip(man["seeds"], man["files"]) if seeds is None or s in seeds]
    load = joblib.load if man["kind"] == "pixel" else (lambda p: tf.keras.models.load_model(p, compile=False))
    return man, [load(bundle_dir / f) for _, f in pairs]


def adapt_batchnorm(model, kind, X, mask, patch=PATCH, n=2000, seed=BASE_SEED):
    """AdaBN: replace the BatchNorm moving statistics with those of the target site (layer by layer, in order)."""
    bns = [l for l in model.layers if isinstance(l, layers.BatchNormalization)]
    if not bns:
        return 0
    rng = np.random.default_rng(seed)
    if kind == "patch":
        r, c = np.nonzero(mask)
        k = rng.choice(len(r), size=min(n, len(r)), replace=False)
        p = patch // 2
        Xpad = np.pad(X, ((p, p), (p, p), (0, 0)), mode="reflect")
        xs = np.stack([Xpad[i:i + patch, j:j + patch] for i, j in zip(r[k], c[k])])
    else:
        coords = [(i, j) for i in window_starts(X.shape[0]) for j in window_starts(X.shape[1])
                  if mask[i:i + SEG_PATCH, j:j + SEG_PATCH].mean() >= MIN_LABEL_FRAC]
        coords = coords or [(i, j) for i in window_starts(X.shape[0]) for j in window_starts(X.shape[1])]
        coords = [coords[i] for i in rng.permutation(len(coords))[:512]]
        xs = np.stack([X[i:i + SEG_PATCH, j:j + SEG_PATCH] for i, j in coords])
    for bn in bns:
        feat = Model(model.input, bn.input).predict(xs, batch_size=256, verbose=0)
        axes = tuple(range(feat.ndim - 1))
        bn.moving_mean.assign(feat.mean(axis=axes).astype("float32"))
        bn.moving_variance.assign(feat.var(axis=axes).astype("float32"))
    return len(bns)


def prepare_site(feature_tif, bands, label_tif=None):
    """Read a site, select and order the bands by name, build the valid mask (and labels if given)."""
    arr, prof, names = read_raster(feature_tif)
    pos = {n.lower(): i for i, n in enumerate(names)}
    missing = [b for b in bands if b.lower() not in pos]
    if missing:
        raise ValueError(f"{Path(feature_tif).name} lacks bands used by the model: {missing}")
    Xt = np.transpose(arr, (1, 2, 0))[..., [pos[b.lower()] for b in bands]]
    v = np.isfinite(Xt).any(axis=-1)
    labels = None
    if label_tif is not None:
        lab = read_raster(label_tif)[0][0]
        if lab.shape != v.shape:
            raise ValueError("Feature and label rasters differ in size.")
        v &= np.isin(lab, [0, 1])
        labels = np.where(v, lab, 0).astype(np.uint8)
    return Xt, v, labels, prof


def standardise_site(Xt, v, man, mode="source"):
    if mode == "source":
        mu, sd = np.asarray(man["band_mean"]), np.asarray(man["band_std"])
    elif mode == "target":
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=RuntimeWarning)
            mu, sd = np.nanmean(Xt[v], axis=0), np.nanstd(Xt[v], axis=0)
        mu = np.where(np.isfinite(mu), mu, 0.0); sd = np.where(np.isfinite(sd) & (sd > 1e-12), sd, 1.0)
    else:
        raise ValueError("mode must be 'source' or 'target'")
    Xs = np.nan_to_num((Xt - mu) / sd, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
    Xs[~v] = 0.0
    return Xs


def predict_bundle(man, models, Xs, v, adabn=False):
    """Seed-ensemble probability (mean of the members)."""
    if man["kind"] == "seg" and man["seg_patch"] != SEG_PATCH:
        raise ValueError(f"Set SEG_PATCH = {man['seg_patch']} (the training value) before transferring this model.")
    probs = []
    for m in models:
        if man["kind"] == "pixel":
            p = np.full(v.shape, np.nan, np.float32); p[v] = m.predict_proba(Xs[v])[:, 1]
        else:
            if adabn:
                adapt_batchnorm(m, man["kind"], Xs, v, patch=man["patch"])
            p = (predict_patch(m, padded(Xs, man["patch"]), v, man["patch"]) if man["kind"] == "patch"
                 else predict_seg(m, Xs, v, tta=man["seg_tta"]))
        probs.append(p)
    return np.mean(probs, axis=0)


def resolve_threshold(prob, v, man, rule="source"):
    if rule == "source":
        return float(man["threshold"])
    if rule == "prevalence":                       # same predicted class-1 share as the training labels; no target labels
        return float(np.quantile(prob[v], 1 - man["train_class1_fraction"]))
    return float(rule)


def transfer_to_site(bundle_dir, feature_tif, label_tif=None, site="new site", standardise="target", adabn=True,
                     threshold="source", seeds=None, out_dir=None, write=True):
    """Apply an exported bundle to another site; maps and GeoTIFFs always, accuracy only when labels are given."""
    man, models = load_bundle(bundle_dir, seeds)
    Xt, v, labels, prof = prepare_site(feature_tif, man["bands"], label_tif)
    Xs = standardise_site(Xt, v, man, standardise)
    prob = predict_bundle(man, models, Xs, v, adabn)
    thr = resolve_threshold(prob, v, man, threshold)
    res = {"model": man["model"], "site": site, "standardise": standardise, "adabn": adabn, "threshold": thr,
           "members": len(models), "pixels": int(v.sum()), "class1_share": float((prob[v] >= thr).mean()),
           "prob": prob, "valid": v}
    if labels is not None:
        res["metrics"] = compute_metrics(labels[v], prob[v], thr)
    if write:
        out = Path(out_dir or OUTPUT_DIR / f"transfer_{slug(site)}"); out.mkdir(parents=True, exist_ok=True)
        tag = f"{man['model']}_{site}"
        save_map_geotiffs(tag, prob, out, labels=labels, valid_mask=v, ref_profile=prof, threshold=thr)
        cls, err = class_and_error_maps(prob, labels if labels is not None else np.zeros(v.shape, np.uint8), v, threshold=thr)
        n = 3 if labels is not None else 2
        fig, axs = plt.subplots(1, n, figsize=(5.4 * n, 4.8))
        im = show_prob(axs[0], prob, v); fig.colorbar(im, ax=axs[0], fraction=0.04, pad=0.02, label="P(higher HI)")
        show_cls(axs[1], cls)
        axs[1].legend(handles=cls_handles(), loc="upper center", bbox_to_anchor=(0.5, -0.02), ncol=2, frameon=False)
        titles = ["(a) Probability", f"(b) Classes (threshold {thr:.3f})"]
        if labels is not None:
            show_err(axs[2], err); titles.append("(c) Errors, whole area")
            axs[2].legend(handles=err_handles(err), loc="upper center", bbox_to_anchor=(0.5, -0.02), ncol=3, frameon=False, fontsize=8)
        for ax, t in zip(axs, titles):
            _map_axes(ax, t)
        fig.suptitle(f"{man['model']} trained at {man['training_site']}, applied to {site} "
                     f"({standardise} statistics{', AdaBN' if adabn else ''})", fontsize=11, fontweight="bold", y=1.0)
        fig.tight_layout()
        _save(fig, f"Fig_transfer_{slug(tag)}", out)
        summary = {k: v_ for k, v_ in res.items() if k not in ("prob", "valid")}
        (out / f"{slug(tag)}_summary.json").write_text(json.dumps(summary, indent=2, default=float))
    return res

The self-check below re-applies the HazardFuse bundle to Glen Lyn with the training settings; the result must match the fitted maps. With labels at the new site, all four standardisation/AdaBN options are scored (T8).

In [ ]:
# (a) Self-check: re-apply the exported HazardFuse bundle to Glen Lyn exactly as it was trained (source statistics,
#     no adaptation). The seed-ensemble probability must equal the mean of the fitted maps.
if EXPORT_MODELS:
    chk = transfer_to_site(BUNDLES[PROPOSED], FEATURE_FILE, LABEL_FILE, site=SITE, standardise="source", adabn=False,
                           write=False)
    fitted = np.mean([r["prob"] for r in RUNS[PROPOSED]], axis=0)
    diff = np.nanmax(np.abs(chk["prob"][valid] - fitted[valid]))
    print(f"Self-check: max |transferred − fitted| = {diff:.2e} over {valid.sum():,} pixels "
          f"({'OK' if diff < 1e-3 else 'CHECK band order / statistics'})")

# (b) New site. Labels are optional; with labels, every standardisation/AdaBN option is scored for comparison.
if NEW_SITE_FEATURES is not None and EXPORT_MODELS:
    TRANSFER = {}
    for name in TRANSFER_MODELS:
        TRANSFER[name] = transfer_to_site(BUNDLES[name], NEW_SITE_FEATURES, NEW_SITE_LABELS, site=NEW_SITE_NAME,
                                          standardise=TRANSFER_STANDARDISE, adabn=TRANSFER_ADABN,
                                          threshold=TRANSFER_THRESHOLD)
        if NEW_SITE_LABELS is not None:
            rows = {}
            for std in ("source", "target"):
                for ada in (False, True):
                    r = transfer_to_site(BUNDLES[name], NEW_SITE_FEATURES, NEW_SITE_LABELS, site=NEW_SITE_NAME,
                                         standardise=std, adabn=ada, threshold=TRANSFER_THRESHOLD, write=False)
                    rows[(std, "AdaBN" if ada else "none")] = {k: r["metrics"][k] for k in
                                                               ("Macro_F1", "MCC", "Kappa", "Balanced_Acc", "ROC_AUC_1")}
                    rows[(std, "AdaBN" if ada else "none")]["Class-1 share"] = r["class1_share"]
            T8 = pd.DataFrame(rows).T.round(3)
            T8.index.names = ["Statistics", "Adaptation"]
            T8.to_csv(TAB_DIR / f"T8_transfer_{slug(name)}_{slug(NEW_SITE_NAME)}.csv")
            print(f"\n{name}: {SITE} -> {NEW_SITE_NAME}"); display(T8)
else:
    print("Set NEW_SITE_FEATURES (and optionally NEW_SITE_LABELS) and EXPORT_MODELS = True to transfer the models.")